# 🧳 مساعد تخطيط السفر الذكي — AI Travel Planning Assistant
**مقرر أساسيات LangChain (LCH303) — المجموعة 12 — الشعبة 1**
جامعة الأمير مساعد بن عبدالرحمن — الكلية التطبيقية

**أعضاء المجموعة:** سلمان سالم الحربي · رهف يوسف الصقر · أمامة محمود حمو · هناء محمد الأفجح · نوال الحربي

---
### فكرة المشروع
مساعد يبني برنامج سفر يومي لثلاث وجهات (جدة، العلا، المدينة المنورة) حسب عدد الأيام والمسافرين والميزانية والاهتمامات.
النموذج اللغوي **يختار** الأنشطة من قاعدة البيانات فقط، ثم **يتحقق** النظام من الخطة، و**يحسب** التكلفة بالكود، ويمكن **تعديل** الخطة بالمحادثة وحفظها.

### طريقة التشغيل
1. أضف مفتاح Groq في **Secrets** (أيقونة المفتاح 🔑 يسار الصفحة) باسم `GROQ_API_KEY` وفعّل الوصول للدفتر.
2. شغّل الخلايا **بالترتيب** من الأعلى للأسفل (أو Runtime ← Run all).

كل خطوة في الدفتر لها: **شرح** ← **خلية الكود** (تُحفظ كملف داخل المشروع) ← **خلية تجربة** تعرض نتيجة الخطوة.

## المرحلة 1: تجهيز البيئة
### الخطوة 1: تثبيت المكتبات
| المكتبة | الاستخدام |
|:---:|:---:|
| `langchain-core` | القوالب، سلاسل LCEL، المحللات، الأدوات، الذاكرة |
| `langchain-groq` | ربط LangChain بنماذج Groq |
| `streamlit` | واجهة المستخدم |
| `pydantic` | المخرجات المنظمة والتحقق |
| `python-dotenv` | قراءة المفاتيح من ملف `.env` عند التشغيل المحلي |
| `pytest` | الاختبارات الآلية |

In [ ]:
!pip install -q "langchain-core>=0.3,<2" "langchain-groq>=0.2,<2" "streamlit>=1.36" "pydantic>=2.7" "python-dotenv>=1.0" "pytest>=8.0"
print("✅ تم تثبيت المكتبات")

### الخطوة 2: إنشاء هيكل المشروع
ننشئ مجلد المشروع والمجلدات الفرعية، ونفعّل إعادة التحميل التلقائي حتى تنعكس أي تعديلات على الملفات مباشرة.

In [ ]:
import os, sys
os.makedirs("/content/travel_planner/src/travel_planner", exist_ok=True)
os.makedirs("/content/travel_planner/tests", exist_ok=True)
os.makedirs("/content/travel_planner/data", exist_ok=True)
os.makedirs("/content/travel_planner/.streamlit", exist_ok=True)
%cd /content/travel_planner
sys.path.insert(0, "/content/travel_planner/src")
%load_ext autoreload
%autoreload 2
print("✅ تم إنشاء هيكل المشروع")

### الخطوة 3: ربط مفتاح Groq بأمان
المفتاح **لا يُكتب داخل الكود أبداً**؛ يُقرأ من أسرار Colab ويُحفظ في متغيرات البيئة فقط.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ تم تحميل المفتاح من Secrets (لن يُطبع لأسباب أمنية)")
except Exception as e:
    print("⚠️ لم يُعثر على المفتاح؛ سيعمل النظام بالخطة الاحتياطية فقط:", type(e).__name__)

## المرحلة 2: الإعدادات والأساسيات
### الخطوة 4: ملف الحزمة والإعدادات (`config.py`)
يجمع كل الإعدادات في مكان واحد: مفتاح API واسم النموذج ومسار قاعدة البيانات وحدود التحقق من المدخلات.

In [ ]:
%%writefile src/travel_planner/__init__.py
"""مساعد تخطيط السفر — AI Travel Planning Assistant (LangChain + Groq + SQLite + Streamlit)."""

__version__ = "1.0.0"


In [ ]:
%%writefile src/travel_planner/config.py
"""
إعدادات التطبيق المركزية.

تُقرأ كل القيم الحساسة (مثل مفتاح Groq) من ملف .env أو من متغيرات البيئة،
ولا يُكتب أي مفتاح داخل الكود إطلاقاً.
"""
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

try:  # python-dotenv اختياري: إن لم يكن مثبتاً نكتفي بمتغيرات البيئة
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:  # pragma: no cover
    pass

# جذر المشروع (المجلد الذي يحتوي app.py)
PROJECT_ROOT = Path(__file__).resolve().parents[2]


@dataclass(frozen=True)
class Settings:
    """قيم الإعداد التي يحتاجها التطبيق."""

    groq_api_key: str | None
    model_name: str
    temperature: float
    max_retries: int
    request_timeout: int
    db_path: Path

    # حدود التحقق من المدخلات
    min_days: int = 1
    max_days: int = 7
    min_travelers: int = 1
    max_travelers: int = 10
    min_budget: float = 100.0
    max_budget: float = 200_000.0
    max_activities_per_day: int = 4

    @property
    def has_api_key(self) -> bool:
        return bool(self.groq_api_key and self.groq_api_key.strip())


def get_settings() -> Settings:
    """يبني كائن الإعدادات من متغيرات البيئة مع قيم افتراضية آمنة."""
    db_path = Path(os.getenv("TRAVEL_DB_PATH", PROJECT_ROOT / "data" / "travel_planner.db"))
    return Settings(
        groq_api_key=os.getenv("GROQ_API_KEY"),
        model_name=os.getenv("GROQ_MODEL", "openai/gpt-oss-120b"),
        temperature=float(os.getenv("LLM_TEMPERATURE", "0.3")),
        max_retries=int(os.getenv("LLM_MAX_RETRIES", "2")),
        request_timeout=int(os.getenv("LLM_TIMEOUT", "60")),
        db_path=db_path,
    )


In [ ]:
# تجربة: قراءة الإعدادات
from travel_planner.config import get_settings
s = get_settings()
print("النموذج:", s.model_name)
print("هل يوجد مفتاح؟", s.has_api_key)
print("قاعدة البيانات:", s.db_path)
print("حدود الأيام:", s.min_days, "إلى", s.max_days)

### الخطوة 5: الأخطاء المخصصة (`exceptions.py`)
كل خطأ يحمل رسالة عربية واضحة تظهر للمستخدم بدلاً من رسائل Python التقنية.

In [ ]:
%%writefile src/travel_planner/exceptions.py
"""
أخطاء مخصّصة للتطبيق.

كل خطأ يحمل رسالة عربية واضحة تُعرض للمستخدم مباشرة في الواجهة،
بدلاً من إظهار تتبّع أخطاء Python التقني.
"""


class TravelPlannerError(Exception):
    """الأصل المشترك لجميع أخطاء التطبيق."""

    user_message = "حدث خطأ غير متوقع. حاول مرة أخرى."

    def __init__(self, message: str | None = None):
        super().__init__(message or self.user_message)
        self.user_message = message or self.user_message


class InvalidInputError(TravelPlannerError):
    """مدخلات غير صالحة من المستخدم (أيام، ميزانية، وجهة...)."""

    user_message = "المدخلات غير صحيحة، راجع الحقول المظللة."


class APIUnavailableError(TravelPlannerError):
    """خدمة النموذج اللغوي غير متاحة (لا يوجد مفتاح، انقطاع، تجاوز الحد)."""

    user_message = "خدمة النموذج اللغوي غير متاحة حالياً."


class DatabaseUnavailableError(TravelPlannerError):
    """تعذّر فتح قاعدة البيانات أو الكتابة فيها."""

    user_message = "قاعدة البيانات غير متاحة حالياً."


class UnsupportedFileError(TravelPlannerError):
    """ملف مرفوع بصيغة غير مدعومة أو بمحتوى تالف."""

    user_message = "صيغة الملف غير مدعومة. يُقبل ملف JSON صادر من التطبيق فقط."


class InvalidModelResponseError(TravelPlannerError):
    """رد النموذج لا يطابق البنية المطلوبة أو يخالف البيانات."""

    user_message = "رد النموذج غير مناسب، أُعيدت المحاولة دون نجاح."


In [ ]:
# تجربة: رسالة الخطأ التي يراها المستخدم
from travel_planner.exceptions import InvalidInputError, APIUnavailableError
for err in (InvalidInputError(), APIUnavailableError("مفتاح Groq غير صحيح")):
    print(type(err).__name__, "←", err.user_message)

### الخطوة 6: نماذج البيانات (`schemas.py`)
نماذج **Pydantic** تحدد شكل الطلب والخطة والميزانية. هي أساس **المخرجات المنظمة** التي يجب أن يلتزم بها النموذج اللغوي.

In [ ]:
%%writefile src/travel_planner/schemas.py
"""
نماذج البيانات (Pydantic) المستخدمة في كامل التطبيق.

- TripRequest: طلب الرحلة القادم من الواجهة، مع قواعد التحقق.
- PlanItem / DayPlan / TravelPlan: البنية التي يجب أن يُرجعها النموذج اللغوي.
  تستخدمها LangChain عبر PydanticOutputParser لإجبار النموذج على مخرجات منظمة.
"""
from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, Field, field_validator

TimeSlot = Literal["صباحاً", "ظهراً", "عصراً", "مساءً"]

# الاهتمامات المدعومة (تطابق عمود category في جدول الأنشطة)
SUPPORTED_INTERESTS = {
    "تاريخ وتراث": "heritage",
    "طبيعة ومغامرة": "nature",
    "تسوق": "shopping",
    "مطاعم ومقاهي": "food",
    "ثقافة وفنون": "culture",
    "ترفيه وعائلة": "family",
}


class TripRequest(BaseModel):
    """طلب الرحلة كما يدخله المستخدم."""

    destination: str = Field(..., description="اسم الوجهة كما في قاعدة البيانات")
    days: int = Field(..., ge=1, le=7)
    travelers: int = Field(..., ge=1, le=10)
    budget: float = Field(..., gt=0, description="الميزانية الإجمالية بالريال")
    interests: list[str] = Field(..., min_length=1)
    notes: str = Field("", max_length=300)

    @field_validator("destination")
    @classmethod
    def _strip_destination(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("يجب اختيار وجهة")
        return v

    @field_validator("interests")
    @classmethod
    def _known_interests(cls, v: list[str]) -> list[str]:
        unknown = [i for i in v if i not in SUPPORTED_INTERESTS]
        if unknown:
            raise ValueError(f"اهتمامات غير مدعومة: {', '.join(unknown)}")
        return v

    @property
    def interest_codes(self) -> list[str]:
        return [SUPPORTED_INTERESTS[i] for i in self.interests]


class PlanItem(BaseModel):
    """نشاط واحد داخل يوم من أيام الرحلة."""

    activity_id: int = Field(..., description="رقم النشاط من قائمة الأنشطة المتاحة فقط")
    activity_name: str = Field(..., description="اسم النشاط كما ورد في القائمة")
    time_slot: TimeSlot = Field(..., description="فترة النشاط: صباحاً أو ظهراً أو عصراً أو مساءً")
    reason: str = Field(..., description="سبب اختيار النشاط في جملة قصيرة")


class DayPlan(BaseModel):
    """برنامج يوم واحد."""

    day_number: int = Field(..., ge=1)
    title: str = Field(..., description="عنوان قصير لليوم")
    items: list[PlanItem] = Field(..., min_length=1, max_length=4)


class TravelPlan(BaseModel):
    """الخطة الكاملة التي يُرجعها النموذج."""

    summary: str = Field(..., description="ملخص الرحلة في جملتين")
    days: list[DayPlan]
    tips: list[str] = Field(default_factory=list, description="نصائح عملية قصيرة (3 كحد أقصى)")


class CostLine(BaseModel):
    """سطر تكلفة محسوب برمجياً (لا يعتمد على النموذج)."""

    label: str
    amount: float


class BudgetReport(BaseModel):
    """نتيجة حساب الميزانية."""

    lines: list[CostLine]
    total: float
    budget: float
    remaining: float
    within_budget: bool
    warnings: list[str] = Field(default_factory=list)


In [ ]:
# تجربة: طلب صحيح وطلب فيه اهتمام غير مدعوم
from travel_planner.schemas import TripRequest
ok = TripRequest(destination="جدة", days=2, travelers=2, budget=4000, interests=["تاريخ وتراث"])
print("✅", ok.destination, ok.interest_codes)
try:
    TripRequest(destination="جدة", days=2, travelers=2, budget=4000, interests=["تزلج"])
except Exception as e:
    print("🛑", str(e).splitlines()[-1][:120])

## المرحلة 3: البيانات
### الخطوة 7: بيانات الوجهات والأنشطة (`seed_data.py`)
ثلاث وجهات و33 نشاطاً سياحياً حقيقياً مع تكاليف تقديرية تجريبية. هذا هو **المصدر الوحيد** الذي يختار منه النموذج.

In [ ]:
%%writefile src/travel_planner/seed_data.py
"""
بيانات تجريبية مُراجعة لثلاث وجهات داخل المملكة.

تنبيه: جميع التكاليف تقديرية لأغراض النموذج الأولي التعليمي فقط،
ولا تمثل أسعاراً رسمية أو محدّثة. التطبيق يعرضها دائماً بوصفها "تقديرية".

الحقول:
- الوجهة: متوسط ليلة الفندق (للغرفة)، الأكل اليومي للفرد، التنقل اليومي للمجموعة.
- النشاط: الفئة، المدة بالساعات، التكلفة للفرد، الفترة المفضلة.
"""

DESTINATIONS = [
    {
        "name": "جدة",
        "description": "عروس البحر الأحمر؛ جدة التاريخية والكورنيش والأسواق.",
        "hotel_night": 420,
        "food_per_person": 110,
        "transport_per_day": 130,
    },
    {
        "name": "العلا",
        "description": "متحف مفتوح؛ آثار الحجر وتكوينات صخرية وأنشطة صحراوية.",
        "hotel_night": 700,
        "food_per_person": 150,
        "transport_per_day": 250,
    },
    {
        "name": "المدينة المنورة",
        "description": "طيبة الطيبة؛ المسجد النبوي والمعالم الإسلامية والتاريخية والمزارع.",
        "hotel_night": 450,
        "food_per_person": 120,
        "transport_per_day": 150,
    },
]

# (اسم النشاط، الفئة، المدة بالساعات، التكلفة للفرد، الفترة المفضلة، وصف)
ACTIVITIES = {
    "جدة": [
        ("جدة التاريخية (البلد)", "heritage", 3, 0, "عصراً", "بيوت الرواشين وموقع تراث عالمي."),
        ("بيت نصيف", "heritage", 1, 20, "عصراً", "من أشهر البيوت التاريخية في البلد."),
        ("كورنيش جدة ونافورة الملك فهد", "family", 2, 0, "مساءً", "ممشى بحري وإطلالة على النافورة."),
        ("مسجد الرحمة (المسجد العائم)", "culture", 1, 0, "عصراً", "مسجد على الماء بإطلالة مميزة."),
        ("رحلة غوص أو سنوركلينغ", "nature", 4, 250, "صباحاً", "شعاب البحر الأحمر المرجانية."),
        ("سوق العلوي", "shopping", 2, 0, "مساءً", "سوق تقليدي للعطور والتوابل."),
        ("رد سي مول", "shopping", 2.5, 0, "مساءً", "مجمع تجاري كبير."),
        ("حديقة الأمير ماجد", "family", 2, 0, "عصراً", "مساحات خضراء وألعاب أطفال."),
        ("مطاعم المأكولات البحرية", "food", 1.5, 120, "ظهراً", "أسماك طازجة على الطريقة الحجازية."),
        ("متحف عبد الرؤوف خليل", "culture", 2, 30, "صباحاً", "مقتنيات تراثية وتاريخية."),
        ("شاطئ أبحر الشمالية", "nature", 3, 50, "ظهراً", "شواطئ وأنشطة مائية."),
    ],
    "العلا": [
        ("الحجر (مدائن صالح)", "heritage", 3, 95, "صباحاً", "مدافن نبطية منحوتة وموقع تراث عالمي."),
        ("جبل الفيل", "nature", 1.5, 0, "مساءً", "تكوين صخري شهير وقت الغروب."),
        ("البلدة القديمة في العلا", "heritage", 2, 0, "عصراً", "أزقة طينية وسوق تراثي."),
        ("دادان وجبل عكمة", "heritage", 2.5, 95, "صباحاً", "آثار مملكتي دادان ولحيان ونقوش عكمة."),
        ("قاعة مرايا", "culture", 1.5, 100, "مساءً", "أكبر مبنى مكسو بالمرايا في العالم (حسب الفعاليات)."),
        ("جولة الدبابات الصحراوية", "nature", 2, 300, "عصراً", "مغامرة بين الكثبان والتكوينات."),
        ("مزرعة النخيل والحمضيات", "family", 1.5, 50, "صباحاً", "جولة في مزارع العلا التقليدية."),
        ("مطل حرة عويرض", "nature", 2, 60, "مساءً", "إطلالة واسعة على الوديان عند الغروب."),
        ("مطاعم البلدة القديمة", "food", 1.5, 110, "ظهراً", "أطباق محلية في أجواء تراثية."),
        ("سوق العلا للمنتجات المحلية", "shopping", 1.5, 0, "عصراً", "تمور ومنتجات حرفية."),
    ],
    "المدينة المنورة": [
        ("زيارة المسجد النبوي", "heritage", 2, 0, "صباحاً", "المسجد النبوي الشريف ومعالمه المعمارية."),
        ("مسجد قباء", "heritage", 1.5, 0, "صباحاً", "أول مسجد بُني في الإسلام."),
        ("جبل أحد ومقبرة الشهداء", "heritage", 2, 0, "عصراً", "موقع غزوة أحد وجبل الرماة."),
        ("المساجد السبعة", "heritage", 1.5, 0, "عصراً", "مساجد تاريخية في موقع غزوة الخندق."),
        ("متحف سكة حديد الحجاز", "culture", 2, 0, "صباحاً", "محطة القطار التاريخية ومقتنياتها."),
        ("متحف دار المدينة", "culture", 2, 50, "عصراً", "تاريخ المدينة المنورة وعمارتها بنماذج مجسمة."),
        ("مجمع الملك فهد لطباعة المصحف الشريف", "culture", 1.5, 0, "صباحاً", "جولة في مراحل طباعة المصحف."),
        ("مزرعة تمور في العالية", "family", 1.5, 40, "صباحاً", "جولة في مزارع النخيل وتذوق التمور."),
        ("وادي الجن (البيضاء)", "nature", 3, 60, "عصراً", "طريق جبلي شمال غرب المدينة بطبيعة صخرية."),
        ("سوق التمور المركزي", "shopping", 1.5, 0, "مساءً", "أنواع التمور المدينية مثل العجوة والعنبرة."),
        ("مجمع النور مول", "shopping", 2.5, 0, "مساءً", "تسوق ومطاعم عائلية."),
        ("مطاعم الأكلات المدينية", "food", 1.5, 90, "ظهراً", "أطباق حجازية ومدينية تقليدية."),
    ],
}


In [ ]:
# تجربة: عدد الأنشطة لكل وجهة
from travel_planner.seed_data import DESTINATIONS, ACTIVITIES
for d in DESTINATIONS:
    print(f"{d['name']:16s} ← {len(ACTIVITIES[d['name']])} نشاطاً")

### الخطوة 8: قاعدة البيانات SQLite (`database.py`)
خمسة جداول: الوجهات، الأنشطة، الرحلات، بنود الخطة، ورسائل المحادثة (ذاكرة دائمة).

In [ ]:
%%writefile src/travel_planner/database.py
"""
طبقة قاعدة البيانات (SQLite).

الجداول:
    destinations   الوجهات وتكاليفها اليومية التقديرية
    activities     الأنشطة، كل نشاط مرتبط بوجهة
    trips          طلبات الرحلات والخطط المحفوظة (الخطة كاملة بصيغة JSON)
    plan_items     عناصر كل خطة (نشاط/يوم/فترة) لتسهيل الاستعلام
    chat_messages  سجل المحادثة لكل رحلة (ذاكرة LangChain للتعديلات)

كل الاستعلامات تستخدم معاملات (?) لمنع حقن SQL.
"""
from __future__ import annotations

import json
import sqlite3
from contextlib import contextmanager
from pathlib import Path
from typing import Iterator

from .exceptions import DatabaseUnavailableError
from .seed_data import ACTIVITIES, DESTINATIONS

SCHEMA = """
CREATE TABLE IF NOT EXISTS destinations (
    id                INTEGER PRIMARY KEY AUTOINCREMENT,
    name              TEXT NOT NULL UNIQUE,
    description       TEXT,
    hotel_night       REAL NOT NULL,
    food_per_person   REAL NOT NULL,
    transport_per_day REAL NOT NULL
);

CREATE TABLE IF NOT EXISTS activities (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    destination_id  INTEGER NOT NULL REFERENCES destinations(id),
    name            TEXT NOT NULL,
    category        TEXT NOT NULL,
    duration_hours  REAL NOT NULL,
    cost_per_person REAL NOT NULL,
    best_time       TEXT NOT NULL,
    description     TEXT
);

CREATE TABLE IF NOT EXISTS trips (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    destination  TEXT NOT NULL,
    days         INTEGER NOT NULL,
    travelers    INTEGER NOT NULL,
    budget       REAL NOT NULL,
    interests    TEXT NOT NULL,
    plan_json    TEXT,
    total_cost   REAL,
    source       TEXT DEFAULT 'llm',
    created_at   TEXT DEFAULT (datetime('now', 'localtime')),
    updated_at   TEXT DEFAULT (datetime('now', 'localtime'))
);

CREATE TABLE IF NOT EXISTS plan_items (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    trip_id      INTEGER NOT NULL REFERENCES trips(id) ON DELETE CASCADE,
    day_number   INTEGER NOT NULL,
    time_slot    TEXT NOT NULL,
    activity_id  INTEGER NOT NULL REFERENCES activities(id)
);

CREATE TABLE IF NOT EXISTS chat_messages (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id  TEXT NOT NULL,
    role        TEXT NOT NULL,
    content     TEXT NOT NULL,
    created_at  TEXT DEFAULT (datetime('now', 'localtime'))
);

CREATE INDEX IF NOT EXISTS idx_activities_dest ON activities(destination_id);
CREATE INDEX IF NOT EXISTS idx_messages_session ON chat_messages(session_id);
"""


class Database:
    """مستودع بيانات بسيط فوق SQLite."""

    def __init__(self, db_path: str | Path):
        self.db_path = Path(db_path)

    # ------------------------------------------------------------------ اتصال
    @contextmanager
    def connect(self) -> Iterator[sqlite3.Connection]:
        """يفتح اتصالاً ويضمن الحفظ أو التراجع وإغلاق الاتصال."""
        try:
            if str(self.db_path) != ":memory:":
                self.db_path.parent.mkdir(parents=True, exist_ok=True)
            conn = sqlite3.connect(self.db_path)
        except (sqlite3.Error, OSError) as exc:
            raise DatabaseUnavailableError(f"تعذّر فتح قاعدة البيانات: {exc}") from exc
        conn.row_factory = sqlite3.Row
        conn.execute("PRAGMA foreign_keys = ON")
        try:
            yield conn
            conn.commit()
        except sqlite3.Error as exc:
            conn.rollback()
            raise DatabaseUnavailableError(f"خطأ في قاعدة البيانات: {exc}") from exc
        finally:
            conn.close()

    def initialize(self) -> None:
        """ينشئ الجداول ويعبّئ البيانات التجريبية عند أول تشغيل فقط."""
        with self.connect() as conn:
            conn.executescript(SCHEMA)
            count = conn.execute("SELECT COUNT(*) FROM destinations").fetchone()[0]
            if count == 0:
                self._seed(conn)

    @staticmethod
    def _seed(conn: sqlite3.Connection) -> None:
        for d in DESTINATIONS:
            cur = conn.execute(
                "INSERT INTO destinations (name, description, hotel_night, food_per_person, transport_per_day)"
                " VALUES (?, ?, ?, ?, ?)",
                (d["name"], d["description"], d["hotel_night"], d["food_per_person"], d["transport_per_day"]),
            )
            dest_id = cur.lastrowid
            conn.executemany(
                "INSERT INTO activities (destination_id, name, category, duration_hours,"
                " cost_per_person, best_time, description) VALUES (?, ?, ?, ?, ?, ?, ?)",
                [(dest_id, *a) for a in ACTIVITIES[d["name"]]],
            )

    # ------------------------------------------------------------- الوجهات
    def list_destinations(self) -> list[dict]:
        with self.connect() as conn:
            rows = conn.execute("SELECT * FROM destinations ORDER BY id").fetchall()
        return [dict(r) for r in rows]

    def get_destination(self, name: str) -> dict | None:
        with self.connect() as conn:
            row = conn.execute("SELECT * FROM destinations WHERE name = ?", (name,)).fetchone()
        return dict(row) if row else None

    # ------------------------------------------------------------- الأنشطة
    def search_activities(self, destination: str, categories: list[str] | None = None) -> list[dict]:
        """يسترجع أنشطة الوجهة، مرتبة بحيث تأتي فئات اهتمام المستخدم أولاً."""
        sql = (
            "SELECT a.* FROM activities a JOIN destinations d ON d.id = a.destination_id"
            " WHERE d.name = ? ORDER BY a.id"
        )
        with self.connect() as conn:
            rows = [dict(r) for r in conn.execute(sql, (destination,)).fetchall()]
        if categories:
            # الأنشطة المطابقة للاهتمامات أولاً، ثم مطاعم (يحتاجها أي برنامج)، ثم البقية
            def rank(a: dict) -> int:
                if a["category"] in categories:
                    return 0
                return 1 if a["category"] == "food" else 2

            rows.sort(key=rank)
        return rows

    def get_activities_by_ids(self, ids: list[int]) -> dict[int, dict]:
        if not ids:
            return {}
        placeholders = ",".join("?" * len(ids))
        with self.connect() as conn:
            rows = conn.execute(f"SELECT * FROM activities WHERE id IN ({placeholders})", ids).fetchall()
        return {r["id"]: dict(r) for r in rows}

    # --------------------------------------------------------------- الرحلات
    def save_trip(self, request: dict, plan: dict, total_cost: float, source: str = "llm",
                  trip_id: int | None = None) -> int:
        """يحفظ رحلة جديدة أو يحدّث رحلة قائمة، مع عناصر الخطة."""
        plan_json = json.dumps(plan, ensure_ascii=False)
        interests = json.dumps(request["interests"], ensure_ascii=False)
        with self.connect() as conn:
            if trip_id is None:
                cur = conn.execute(
                    "INSERT INTO trips (destination, days, travelers, budget, interests, plan_json, total_cost, source)"
                    " VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                    (request["destination"], request["days"], request["travelers"], request["budget"],
                     interests, plan_json, total_cost, source),
                )
                trip_id = cur.lastrowid
            else:
                conn.execute(
                    "UPDATE trips SET plan_json = ?, total_cost = ?, source = ?,"
                    " updated_at = datetime('now', 'localtime') WHERE id = ?",
                    (plan_json, total_cost, source, trip_id),
                )
                conn.execute("DELETE FROM plan_items WHERE trip_id = ?", (trip_id,))
            conn.executemany(
                "INSERT INTO plan_items (trip_id, day_number, time_slot, activity_id) VALUES (?, ?, ?, ?)",
                [
                    (trip_id, day["day_number"], item["time_slot"], item["activity_id"])
                    for day in plan["days"]
                    for item in day["items"]
                ],
            )
        return trip_id

    def list_trips(self) -> list[dict]:
        with self.connect() as conn:
            rows = conn.execute(
                "SELECT id, destination, days, travelers, budget, total_cost, source, created_at, updated_at"
                " FROM trips ORDER BY id DESC"
            ).fetchall()
        return [dict(r) for r in rows]

    def get_trip(self, trip_id: int) -> dict | None:
        with self.connect() as conn:
            row = conn.execute("SELECT * FROM trips WHERE id = ?", (trip_id,)).fetchone()
        if not row:
            return None
        trip = dict(row)
        trip["interests"] = json.loads(trip["interests"])
        trip["plan"] = json.loads(trip["plan_json"]) if trip["plan_json"] else None
        return trip

    def delete_trip(self, trip_id: int) -> None:
        with self.connect() as conn:
            conn.execute("DELETE FROM chat_messages WHERE session_id = ?", (f"trip-{trip_id}",))
            conn.execute("DELETE FROM trips WHERE id = ?", (trip_id,))

    # -------------------------------------------------------------- المحادثة
    def add_message(self, session_id: str, role: str, content: str) -> None:
        with self.connect() as conn:
            conn.execute(
                "INSERT INTO chat_messages (session_id, role, content) VALUES (?, ?, ?)",
                (session_id, role, content),
            )

    def get_messages(self, session_id: str) -> list[dict]:
        with self.connect() as conn:
            rows = conn.execute(
                "SELECT role, content FROM chat_messages WHERE session_id = ? ORDER BY id", (session_id,)
            ).fetchall()
        return [dict(r) for r in rows]

    def clear_messages(self, session_id: str) -> None:
        with self.connect() as conn:
            conn.execute("DELETE FROM chat_messages WHERE session_id = ?", (session_id,))


In [ ]:
# تجربة: إنشاء القاعدة وعرض الجداول
import sqlite3, pandas as pd
from travel_planner.database import Database
db = Database(s.db_path)
db.initialize()
con = sqlite3.connect(s.db_path)
for t in ["destinations", "activities", "trips", "plan_items", "chat_messages"]:
    print(f"{t:15s}", con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], "صف")
display(pd.read_sql("SELECT id, name, hotel_night, food_per_person, transport_per_day FROM destinations", con))

In [ ]:
# تجربة: استرجاع أنشطة المدينة المنورة المرتبطة بالتراث (Retrieval)
pd.DataFrame(db.search_activities("المدينة المنورة", ["heritage"]))[["id", "name", "category", "cost_per_person", "best_time"]]

## المرحلة 4: المنطق البرمجي
### الخطوة 9: حساب الميزانية (`budget.py`)
مبدأ مهم: **النموذج لا يحسب الأرقام**. التكلفة تُحسب هنا بالكود من بيانات القاعدة، فتكون دقيقة دائماً.

In [ ]:
%%writefile src/travel_planner/budget.py
"""
حساب الميزانية برمجياً.

مبدأ مهم في المشروع: النموذج اللغوي لا يحسب الأرقام ولا يخترع الأسعار.
النموذج يختار الأنشطة فقط، ثم تُحسب التكلفة هنا من قاعدة البيانات.
"""
from __future__ import annotations

import math

from .schemas import BudgetReport, CostLine


def rooms_needed(travelers: int) -> int:
    """غرفة لكل شخصين (تقريب للأعلى)."""
    return max(1, math.ceil(travelers / 2))


def nights_for(days: int) -> int:
    """رحلة يوم واحد بلا مبيت، وإلا فعدد الليالي = الأيام - 1."""
    return max(0, days - 1)


def base_cost(destination: dict, days: int, travelers: int) -> float:
    """التكلفة الأساسية (سكن + أكل + تنقل) دون الأنشطة."""
    hotel = destination["hotel_night"] * rooms_needed(travelers) * nights_for(days)
    food = destination["food_per_person"] * travelers * days
    transport = destination["transport_per_day"] * days
    return float(hotel + food + transport)


def calculate_budget(destination: dict, days: int, travelers: int, budget: float,
                     activities: list[dict]) -> BudgetReport:
    """
    يحسب التكلفة التفصيلية للخطة.

    activities: قائمة الأنشطة المختارة (كما هي في قاعدة البيانات، بها cost_per_person).
    """
    rooms = rooms_needed(travelers)
    nights = nights_for(days)
    hotel = destination["hotel_night"] * rooms * nights
    food = destination["food_per_person"] * travelers * days
    transport = destination["transport_per_day"] * days
    acts = sum(a["cost_per_person"] for a in activities) * travelers

    lines = [
        CostLine(label=f"السكن ({nights} ليلة × {rooms} غرفة)", amount=round(hotel, 2)),
        CostLine(label=f"الوجبات ({travelers} مسافر × {days} يوم)", amount=round(food, 2)),
        CostLine(label=f"التنقل الداخلي ({days} يوم)", amount=round(transport, 2)),
        CostLine(label=f"تذاكر الأنشطة ({len(activities)} نشاط × {travelers} مسافر)", amount=round(acts, 2)),
    ]
    total = round(sum(line.amount for line in lines), 2)
    remaining = round(budget - total, 2)

    warnings: list[str] = []
    if remaining < 0:
        warnings.append(
            f"التكلفة التقديرية تتجاوز الميزانية بمقدار {abs(remaining):,.0f} ريال. "
            "يمكنك طلب خطة أوفر أو زيادة الميزانية."
        )
    elif remaining > budget * 0.4:
        warnings.append("يتبقى جزء كبير من الميزانية؛ يمكنك طلب أنشطة إضافية.")
    free = [a["name"] for a in activities if a["cost_per_person"] == 0]
    if free:
        warnings.append(f"{len(free)} من الأنشطة مجانية الدخول وفق البيانات المتاحة.")

    return BudgetReport(
        lines=lines,
        total=total,
        budget=budget,
        remaining=remaining,
        within_budget=remaining >= 0,
        warnings=warnings,
    )


In [ ]:
# تجربة: تكلفة 3 أيام في المدينة لشخصين مع أول 4 أنشطة
from travel_planner.budget import calculate_budget
dest = db.get_destination("المدينة المنورة")
acts = db.search_activities("المدينة المنورة")[:4]
report = calculate_budget(dest, days=3, travelers=2, budget=5000, activities=acts)
for line in report.lines:
    print(f"{line.label:40s} {line.amount:>8,.0f} ريال")
print(f"الإجمالي: {report.total:,.0f} | المتبقي: {report.remaining:,.0f} | ضمن الميزانية؟ {report.within_budget}")

### الخطوة 10: التحقق (`validation.py`)
طبقتان من التحقق:
1. **التحقق من المدخلات** قبل إرسالها للنموذج.
2. **التحقق من خطة النموذج**: رفض أي نشاط غير موجود في القاعدة، ومنع التكرار، وضبط عدد الأيام.

In [ ]:
%%writefile src/travel_planner/validation.py
"""
التحقق من المدخلات ومن مخرجات النموذج.

1) validate_request: يتحقق من طلب المستخدم قبل أي استدعاء للنموذج.
2) validate_plan:    يتحقق من خطة النموذج مقابل الطلب وقاعدة البيانات
                     (عدد الأيام، أرقام الأنشطة، التكرار، الحد اليومي)
                     حتى لا تُعرض أنشطة مختلقة.
"""
from __future__ import annotations

from pydantic import ValidationError

from .budget import base_cost
from .config import Settings
from .database import Database
from .exceptions import InvalidInputError
from .schemas import TravelPlan, TripRequest

_FIELD_NAMES = {
    "destination": "الوجهة",
    "days": "عدد الأيام",
    "travelers": "عدد المسافرين",
    "budget": "الميزانية",
    "interests": "الاهتمامات",
    "notes": "الملاحظات",
}


def validate_request(raw: dict, db: Database, settings: Settings) -> tuple[TripRequest, list[str]]:
    """
    يرجع (الطلب بعد التحقق، قائمة تحذيرات).
    يرفع InvalidInputError برسالة عربية تجمع كل الأخطاء.
    """
    errors: list[str] = []

    # تحويل الأنواع بأمان (قد تصل كنصوص من واجهة أو ملف)
    for key, caster in (("days", int), ("travelers", int), ("budget", float)):
        if key in raw:
            try:
                raw[key] = caster(raw[key])
            except (TypeError, ValueError):
                errors.append(f"{_FIELD_NAMES[key]}: يجب أن يكون رقماً.")

    if errors:
        raise InvalidInputError("\n".join(errors))

    try:
        request = TripRequest(**raw)
    except ValidationError as exc:
        for err in exc.errors():
            field = _FIELD_NAMES.get(str(err["loc"][0]) if err["loc"] else "", "حقل")
            errors.append(f"{field}: {err['msg']}")
        raise InvalidInputError("\n".join(errors)) from exc

    # قواعد العمل الإضافية
    if not settings.min_days <= request.days <= settings.max_days:
        errors.append(f"عدد الأيام يجب أن يكون بين {settings.min_days} و{settings.max_days}.")
    if not settings.min_travelers <= request.travelers <= settings.max_travelers:
        errors.append(f"عدد المسافرين يجب أن يكون بين {settings.min_travelers} و{settings.max_travelers}.")
    if not settings.min_budget <= request.budget <= settings.max_budget:
        errors.append(f"الميزانية يجب أن تكون بين {settings.min_budget:,.0f} و{settings.max_budget:,.0f} ريال.")

    destination = db.get_destination(request.destination)
    if destination is None:
        names = "، ".join(d["name"] for d in db.list_destinations())
        errors.append(f"الوجهة «{request.destination}» غير متوفرة. الوجهات المتاحة: {names}.")

    if errors:
        raise InvalidInputError("\n".join(errors))

    warnings: list[str] = []
    minimum = base_cost(destination, request.days, request.travelers)
    if request.budget < minimum:
        warnings.append(
            f"الميزانية ({request.budget:,.0f} ريال) أقل من التكلفة الأساسية التقديرية للسكن والأكل "
            f"والتنقل ({minimum:,.0f} ريال)؛ ستُقترح أنشطة مجانية قدر الإمكان."
        )
    return request, warnings


def validate_plan(plan: TravelPlan, request: TripRequest, available: dict[int, dict],
                  max_per_day: int = 4) -> list[str]:
    """
    يرجع قائمة بالمشكلات الجوهرية في خطة النموذج (قائمة فارغة = خطة سليمة).
    available: الأنشطة المسموح بها لهذه الوجهة {id: activity}.
    يصحح أيضاً أسماء الأنشطة لتطابق قاعدة البيانات.
    """
    issues: list[str] = []

    if len(plan.days) != request.days:
        issues.append(f"عدد الأيام في الخطة {len(plan.days)} بينما المطلوب {request.days}.")

    numbers = sorted(d.day_number for d in plan.days)
    if numbers != list(range(1, len(plan.days) + 1)):
        issues.append("ترقيم الأيام غير متسلسل.")

    seen: set[int] = set()
    for day in plan.days:
        if len(day.items) > max_per_day:
            issues.append(f"اليوم {day.day_number} يحتوي أكثر من {max_per_day} أنشطة.")
        for item in day.items:
            if item.activity_id not in available:
                issues.append(f"النشاط رقم {item.activity_id} ({item.activity_name}) غير موجود في بيانات الوجهة.")
                continue
            if item.activity_id in seen:
                issues.append(f"النشاط «{available[item.activity_id]['name']}» مكرر في أكثر من موضع.")
            seen.add(item.activity_id)
            # الاسم الرسمي من قاعدة البيانات يمنع أي تحريف من النموذج
            item.activity_name = available[item.activity_id]["name"]

    return issues


In [ ]:
# تجربة: مدخلات خاطئة تجمع كل الأخطاء في رسالة واحدة
from travel_planner.validation import validate_request
from travel_planner.exceptions import TravelPlannerError
try:
    validate_request({"destination": "باريس", "days": 12, "travelers": 2, "budget": -5, "interests": ["تسوق"]}, db, s)
except TravelPlannerError as e:
    print("🛑", e.user_message)

### الخطوة 11: التنسيق (`formatting.py`)
يحوّل الأنشطة المسترجعة إلى جدول نصي يُرسل للنموذج، ويجهّز نسخة نصية من الخطة للتصدير.

In [ ]:
%%writefile src/travel_planner/formatting.py
"""
دوال تنسيق نصية (بدون LangChain) لتجهيز السياق للنموذج وللتصدير.
"""
from __future__ import annotations

from .schemas import BudgetReport, TravelPlan

CATEGORY_AR = {
    "heritage": "تاريخ وتراث",
    "nature": "طبيعة ومغامرة",
    "shopping": "تسوق",
    "food": "مطاعم ومقاهي",
    "culture": "ثقافة وفنون",
    "family": "ترفيه وعائلة",
}


def activities_table(activities: list[dict]) -> str:
    """جدول Markdown مضغوط بالأنشطة؛ هو المصدر الوحيد الذي يختار منه النموذج."""
    header = "| الرقم | النشاط | الفئة | المدة (ساعة) | التكلفة للفرد (ريال) | الوقت المفضل |\n|---|---|---|---|---|---|"
    rows = [
        f"| {a['id']} | {a['name']} | {CATEGORY_AR.get(a['category'], a['category'])} | "
        f"{a['duration_hours']:g} | {a['cost_per_person']:g} | {a['best_time']} |"
        for a in activities
    ]
    return "\n".join([header, *rows])


def plan_to_text(plan: TravelPlan, request: dict, budget: BudgetReport | None,
                 activities: dict[int, dict] | None = None) -> str:
    """نسخة نصية من الخطة للتصدير (ملف .txt)."""
    activities = activities or {}
    lines = [
        "مساعد تخطيط السفر — برنامج الرحلة",
        "=" * 40,
        f"الوجهة: {request['destination']} | الأيام: {request['days']} | "
        f"المسافرون: {request['travelers']} | الميزانية: {request['budget']:,.0f} ريال",
        "",
        plan.summary,
        "",
    ]
    for day in plan.days:
        lines.append(f"اليوم {day.day_number}: {day.title}")
        for item in day.items:
            act = activities.get(item.activity_id, {})
            cost = act.get("cost_per_person")
            cost_txt = f" — {cost:g} ريال/فرد" if cost is not None else ""
            lines.append(f"  • {item.time_slot}: {item.activity_name}{cost_txt}")
            lines.append(f"      {item.reason}")
        lines.append("")
    if budget:
        lines.append("التكلفة التقديرية")
        lines.append("-" * 40)
        for line in budget.lines:
            lines.append(f"  {line.label}: {line.amount:,.0f} ريال")
        lines.append(f"  الإجمالي: {budget.total:,.0f} ريال | المتبقي: {budget.remaining:,.0f} ريال")
        for w in budget.warnings:
            lines.append(f"  ! {w}")
        lines.append("")
    if plan.tips:
        lines.append("نصائح")
        lines.extend(f"  - {t}" for t in plan.tips)
    lines.append("")
    lines.append("تنبيه: جميع التكاليف تقديرية من بيانات تجريبية ولا تمثل أسعاراً رسمية.")
    return "\n".join(lines)


In [ ]:
# تجربة: الجدول الذي يراه النموذج
from travel_planner.formatting import activities_table
print(activities_table(db.search_activities("العلا", ["nature"])[:5]))

## المرحلة 5: مكونات LangChain
### الخطوة 12: قوالب التعليمات — Prompt Engineering (`prompts.py`)
قالب `ChatPromptTemplate` يحدد دور النموذج، ويثبّت السياق على جدول الأنشطة فقط، ويفرض قيوداً مرقمة وصيغة إخراج JSON.

In [ ]:
%%writefile src/travel_planner/prompts.py
"""
قوالب التعليمات (Prompt Engineering).

مبادئ التصميم:
1. دور واضح للنموذج (مخطط رحلات سعودي محترف).
2. سياق مُسترجع من قاعدة البيانات فقط (Grounding): جدول أنشطة مرقّم.
3. قيود صريحة: عدد الأيام، الحد اليومي، عدم التكرار، الالتزام بالأرقام.
4. منع الاختلاق: ممنوع ذكر أي نشاط أو سعر غير موجود في الجدول.
5. لا حساب أرقام: التكلفة تُحسب برمجياً، والنموذج يركّز على الاختيار والترتيب.
6. مخرجات منظمة: تعليمات تنسيق JSON يولّدها PydanticOutputParser تلقائياً.
7. تغذية راجعة: متغير {feedback} يحمل أخطاء المحاولة السابقة عند إعادة المحاولة.
"""
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM_PROMPT = """أنت "رفيق"، مساعد ذكي محترف في تخطيط الرحلات داخل المملكة العربية السعودية.
مهمتك بناء برنامج سفر يومي واقعي ومتوازن بناءً على بيانات الأنشطة المرفقة فقط.

القواعد الإلزامية:
1. استخدم فقط الأنشطة الموجودة في "جدول الأنشطة المتاحة"، واكتب رقمها (activity_id) واسمها كما هما تماماً.
2. لا تخترع أنشطة أو أماكن أو أسعاراً أو مواعيد غير موجودة في الجدول.
3. عدد الأيام في الخطة يساوي عدد أيام الرحلة المطلوب بالضبط، مرقمة من 1.
4. من 2 إلى {max_per_day} أنشطة لكل يوم، ولا تكرر النشاط نفسه في الرحلة.
5. رتّب أنشطة اليوم حسب الفترة (صباحاً ثم ظهراً ثم عصراً ثم مساءً) واستفد من "الوقت المفضل" لكل نشاط.
6. قدّم الأنشطة المطابقة لاهتمامات المسافر، وأضف وجبة أو مطعماً في معظم الأيام إن توفر.
7. راعِ الميزانية: إذا كانت محدودة ففضّل الأنشطة المجانية والأقل تكلفة. لا تحسب المجاميع؛ النظام يحسبها.
8. إذا كانت الأنشطة المتاحة أقل من المطلوب، قلّل عدد الأنشطة اليومية بدلاً من التكرار أو الاختلاق.
9. اكتب بالعربية الفصحى المبسطة، وبإيجاز.
10. أعد المخرجات بصيغة JSON فقط وفق التنسيق المطلوب، دون أي نص قبله أو بعده."""

PLANNING_HUMAN = """## بيانات الرحلة
- الوجهة: {destination} — {destination_description}
- عدد الأيام: {days}
- عدد المسافرين: {travelers}
- الميزانية الإجمالية: {budget} ريال
- التكلفة الأساسية التقديرية (سكن + أكل + تنقل): {base_cost} ريال
- المتبقي تقريباً للأنشطة: {activities_budget} ريال
- الاهتمامات: {interests}
- ملاحظات المسافر: {notes}

## جدول الأنشطة المتاحة (المصدر الوحيد المسموح به)
{activities_table}

{feedback}

## تنسيق المخرجات
{format_instructions}"""

REVISION_HUMAN = """## سياق الرحلة الحالية
{trip_context}

## الخطة الحالية (JSON)
{current_plan}

## جدول الأنشطة المتاحة (المصدر الوحيد المسموح به)
{activities_table}

## طلب التعديل من المسافر
{request}

عدّل الخطة لتلبية الطلب مع الالتزام بجميع القواعد، وأبقِ ما لم يطلب المسافر تغييره كما هو.
إذا كان الطلب غير ممكن بالبيانات المتاحة فاذكر ذلك في الملخص وأعد أقرب خطة ممكنة.

{feedback}

## تنسيق المخرجات
{format_instructions}"""


def planning_prompt() -> ChatPromptTemplate:
    """قالب إنشاء خطة جديدة."""
    return ChatPromptTemplate.from_messages(
        [("system", SYSTEM_PROMPT), ("human", PLANNING_HUMAN)]
    )


def revision_prompt() -> ChatPromptTemplate:
    """قالب تعديل خطة قائمة؛ يتضمن سجل المحادثة السابق (الذاكرة)."""
    return ChatPromptTemplate.from_messages(
        [
            ("system", SYSTEM_PROMPT),
            MessagesPlaceholder(variable_name="history"),
            ("human", REVISION_HUMAN),
        ]
    )


### الخطوة 13: أدوات LangChain (`tools.py`)
ثلاث أدوات `StructuredTool`: **البحث** عن الأنشطة، **حساب الميزانية**، و**حفظ الخطة**.

In [ ]:
%%writefile src/travel_planner/tools.py
"""
أدوات LangChain (Tools).

ثلاث أدوات تغلّف منطق قاعدة البيانات والحساب:
    search_activities  استرجاع الأنشطة المناسبة من SQLite (Retrieval)
    calculate_budget   حساب التكلفة التفصيلية ومقارنتها بالميزانية
    save_plan          حفظ الخطة في قاعدة البيانات

تُستدعى الأدوات داخل سلسلة المعالجة في مراحل محددة (وليس بقرار حر من النموذج)،
وهذا يجعل السلوك متوقعاً وقابلاً للاختبار، كما ورد في مقترح المشروع.
"""
from __future__ import annotations

from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

from .budget import calculate_budget as _calculate_budget
from .database import Database


class SearchActivitiesInput(BaseModel):
    destination: str = Field(..., description="اسم الوجهة")
    categories: list[str] = Field(default_factory=list, description="رموز فئات الاهتمام")


class CalculateBudgetInput(BaseModel):
    destination: str
    days: int
    travelers: int
    budget: float
    activity_ids: list[int]


class SavePlanInput(BaseModel):
    request: dict
    plan: dict
    total_cost: float
    source: str = "llm"
    trip_id: int | None = None


def make_tools(db: Database) -> dict[str, StructuredTool]:
    """ينشئ الأدوات مرتبطة بقاعدة بيانات محددة."""

    def search_activities(destination: str, categories: list[str] | None = None) -> list[dict]:
        """يسترجع أنشطة الوجهة من قاعدة البيانات مرتبة حسب الاهتمامات."""
        return db.search_activities(destination, categories or [])

    def calculate_budget(destination: str, days: int, travelers: int, budget: float,
                         activity_ids: list[int]) -> dict:
        """يحسب تكلفة الرحلة التقديرية من قاعدة البيانات ويقارنها بالميزانية."""
        dest = db.get_destination(destination)
        acts = list(db.get_activities_by_ids(activity_ids).values())
        return _calculate_budget(dest, days, travelers, budget, acts).model_dump()

    def save_plan(request: dict, plan: dict, total_cost: float, source: str = "llm",
                  trip_id: int | None = None) -> int:
        """يحفظ الخطة (جديدة أو محدّثة) ويرجع رقم الرحلة."""
        return db.save_trip(request, plan, total_cost, source=source, trip_id=trip_id)

    return {
        "search_activities": StructuredTool.from_function(
            func=search_activities, name="search_activities",
            description="البحث عن الأنشطة السياحية لوجهة معينة في قاعدة البيانات",
            args_schema=SearchActivitiesInput,
        ),
        "calculate_budget": StructuredTool.from_function(
            func=calculate_budget, name="calculate_budget",
            description="حساب التكلفة التقديرية للرحلة ومقارنتها بالميزانية",
            args_schema=CalculateBudgetInput,
        ),
        "save_plan": StructuredTool.from_function(
            func=save_plan, name="save_plan",
            description="حفظ خطة الرحلة في قاعدة البيانات",
            args_schema=SavePlanInput,
        ),
    }


In [ ]:
# تجربة: استدعاء أداة البحث كما يستدعيها النظام
from travel_planner.tools import make_tools
tools = make_tools(db)
print("الأدوات:", list(tools))
found = tools["search_activities"].invoke({"destination": "جدة", "categories": ["food"]})
print("نتائج البحث:", [a["name"] for a in found[:5]])

### الخطوة 14: ربط النموذج اللغوي (`llm.py`)
ينشئ `ChatGroq` عبر LangChain مع إعادة محاولة تلقائية ومهلة، ويترجم أخطاء الخدمة (مفتاح خاطئ، تجاوز الحد، انقطاع) لرسائل عربية.

In [ ]:
%%writefile src/travel_planner/llm.py
"""
ربط النموذج اللغوي (Groq) عبر LangChain، وتحويل أخطاء الخدمة إلى رسائل مفهومة.
"""
from __future__ import annotations

from .config import Settings
from .exceptions import APIUnavailableError


def build_llm(settings: Settings):
    """ينشئ ChatGroq. يرفع APIUnavailableError إذا لم يوجد مفتاح أو المكتبة."""
    if not settings.has_api_key:
        raise APIUnavailableError(
            "لم يُعثر على مفتاح GROQ_API_KEY. أضفه في ملف .env أو في أسرار Colab."
        )
    try:
        from langchain_groq import ChatGroq
    except ImportError as exc:  # pragma: no cover
        raise APIUnavailableError("مكتبة langchain-groq غير مثبتة. شغّل: pip install -r requirements.txt") from exc

    return ChatGroq(
        model=settings.model_name,
        temperature=settings.temperature,
        max_retries=settings.max_retries,  # إعادة المحاولة التلقائية عند أخطاء الشبكة المؤقتة
        timeout=settings.request_timeout,
        api_key=settings.groq_api_key,
    )


def translate_llm_error(exc: Exception) -> APIUnavailableError:
    """
    يحوّل أخطاء مكتبة Groq/الشبكة إلى رسالة عربية واضحة.
    نعتمد على اسم الصنف حتى لا نربط الكود بإصدار معين من المكتبة.
    """
    name = type(exc).__name__
    if "Authentication" in name or "401" in str(exc):
        return APIUnavailableError("مفتاح Groq غير صحيح أو منتهي الصلاحية.")
    if "RateLimit" in name or "429" in str(exc):
        return APIUnavailableError("تم تجاوز حد الطلبات في Groq. انتظر دقيقة ثم أعد المحاولة.")
    if "Connection" in name or "Timeout" in name:
        return APIUnavailableError("تعذّر الاتصال بخدمة Groq. تحقق من الإنترنت.")
    if "NotFound" in name or ("model" in str(exc).lower() and "not found" in str(exc).lower()):
        return APIUnavailableError("اسم النموذج غير متاح في Groq. غيّر GROQ_MODEL في ملف .env.")
    return APIUnavailableError(f"خطأ من خدمة النموذج: {name}")


In [ ]:
# تجربة: سؤال مباشر للنموذج عبر LangChain
from travel_planner.llm import build_llm
try:
    llm = build_llm(s)
    print(llm.invoke("عرّف نفسك في جملة واحدة بالعربية").content)
except TravelPlannerError as e:
    print("⚠️", e.user_message)

### الخطوة 15: السلاسل والذاكرة (`chains.py`)
- **سلسلة التخطيط (LCEL):** استرجاع ← قالب ← نموذج ← تنظيف ← محلل Pydantic.
- **سلسلة التعديل:** نفس الفكرة مع `RunnableWithMessageHistory` وذاكرة محفوظة في SQLite.

In [ ]:
%%writefile src/travel_planner/chains.py
"""
سلاسل LangChain (LCEL) والذاكرة.

سلسلة الإنشاء:
    المدخلات ─► [Retrieval: أداة search_activities] ─► ChatPromptTemplate ─► ChatGroq ─► PydanticOutputParser ─► TravelPlan

سلسلة التعديل:
    RunnableWithMessageHistory( revision_prompt | ChatGroq ) ─► PydanticOutputParser
    حيث يُحفظ سجل المحادثة في SQLite عبر SQLiteChatHistory، فيتذكر النموذج
    طلبات التعديل السابقة لنفس الرحلة.
"""
from __future__ import annotations

from typing import Sequence

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import Runnable, RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory

from .database import Database
from .formatting import activities_table
from .prompts import planning_prompt, revision_prompt
from .schemas import TravelPlan


# ---------------------------------------------------------------- الذاكرة
class SQLiteChatHistory(BaseChatMessageHistory):
    """سجل محادثة دائم في جدول chat_messages، مفتاحه رقم الجلسة (الرحلة)."""

    def __init__(self, db: Database, session_id: str, max_messages: int = 6):
        self.db = db
        self.session_id = session_id
        self.max_messages = max_messages  # آخر الرسائل فقط حتى لا يطول السياق

    @property
    def messages(self) -> list[BaseMessage]:  # type: ignore[override]
        rows = self.db.get_messages(self.session_id)[-self.max_messages:]
        return [HumanMessage(r["content"]) if r["role"] == "human" else AIMessage(r["content"]) for r in rows]

    def add_messages(self, messages: Sequence[BaseMessage]) -> None:
        for m in messages:
            role = "human" if m.type == "human" else "ai"
            content = m.content if isinstance(m.content, str) else str(m.content)
            if role == "ai":
                # نحفظ ملخصاً مختصراً لرد النموذج بدل JSON كاملاً لتوفير السياق
                content = _summarize_ai_plan(content)
            self.db.add_message(self.session_id, role, content)

    def clear(self) -> None:
        self.db.clear_messages(self.session_id)


def _summarize_ai_plan(raw: str) -> str:
    try:
        plan = TravelPlan.model_validate_json(_extract_json(raw))
        days = "؛ ".join(
            f"اليوم {d.day_number}: " + "، ".join(i.activity_name for i in d.items) for d in plan.days
        )
        return f"عدّلتُ الخطة. {plan.summary} ({days})"
    except Exception:  # الرد ليس JSON صالحاً
        return raw[:500]


def _extract_json(text: str) -> str:
    """يقتطع كتلة JSON من الرد حتى لو أحاطها النموذج بعلامات ```."""
    start, end = text.find("{"), text.rfind("}")
    return text[start:end + 1] if start != -1 and end != -1 else text


# ---------------------------------------------------------------- المحلل
def plan_parser() -> PydanticOutputParser:
    return PydanticOutputParser(pydantic_object=TravelPlan)


def _clean_llm_output(message: AIMessage) -> str:
    """تنظيف بسيط قبل التحليل: إزالة أي نص خارج كتلة JSON."""
    return _extract_json(message.content if isinstance(message.content, str) else str(message.content))


# ---------------------------------------------------------- سلسلة الإنشاء
def build_planning_chain(llm, search_tool) -> Runnable:
    """
    سلسلة LCEL كاملة: استرجاع ─► قالب ─► نموذج ─► تنظيف ─► محلل Pydantic.
    search_tool: أداة LangChain (search_activities) تُستدعى أولاً لجلب السياق.
    """
    parser = plan_parser()

    def retrieve(inputs: dict) -> str:
        acts = search_tool.invoke({"destination": inputs["destination"], "categories": inputs["categories"]})
        return activities_table(acts)

    return (
        RunnablePassthrough.assign(
            activities_table=RunnableLambda(retrieve),
            format_instructions=lambda _: parser.get_format_instructions(),
        )
        | planning_prompt()
        | llm
        | RunnableLambda(_clean_llm_output)
        | parser
    )


# ---------------------------------------------------------- سلسلة التعديل
def build_revision_chain(llm, db: Database) -> Runnable:
    """سلسلة تعديل الخطة مع ذاكرة محادثة دائمة لكل رحلة."""
    parser = plan_parser()
    core = revision_prompt() | llm  # يخرج AIMessage حتى تحفظه الذاكرة

    with_history = RunnableWithMessageHistory(
        core,
        lambda session_id: SQLiteChatHistory(db, session_id),
        input_messages_key="request",
        history_messages_key="history",
    )
    return (
        RunnablePassthrough.assign(format_instructions=lambda _: parser.get_format_instructions())
        | with_history
        | RunnableLambda(_clean_llm_output)
        | parser
    )


In [ ]:
# تجربة: القالب بعد تعبئته كما يصل للنموذج
from travel_planner.prompts import planning_prompt
from travel_planner.chains import plan_parser
acts = db.search_activities("جدة", ["heritage", "food"])
messages = planning_prompt().format_messages(
    destination="جدة", destination_description="عروس البحر الأحمر", days=2, travelers=2,
    budget="4,000", base_cost="1,300", activities_budget="2,700", interests="تاريخ وتراث، مطاعم ومقاهي",
    notes="لا يوجد", max_per_day=4, activities_table=activities_table(acts),
    feedback="", format_instructions=plan_parser().get_format_instructions())
for m in messages:
    print(f"===== {m.type} =====\n{m.content[:1200]}\n")

### الخطوة 16: المخطط الاحتياطي (`fallback.py`)
إذا تعذّر الوصول للنموذج (انقطاع، لا يوجد مفتاح)، يبني خطة مبسطة بالقواعد حتى لا يتوقف التطبيق أثناء العرض.

In [ ]:
%%writefile src/travel_planner/fallback.py
"""
مخطِّط احتياطي قائم على القواعد (بدون نموذج لغوي).

يُستخدم فقط عند تعذّر الوصول إلى خدمة النموذج (انقطاع الإنترنت، عدم وجود مفتاح،
تجاوز حد الطلبات)، حتى يبقى التطبيق قابلاً للاستخدام أثناء العرض.
الخطة الناتجة تُعلَّم بالمصدر "fallback" ويظهر للمستخدم أنها خطة مبسطة.
"""
from __future__ import annotations

from .schemas import DayPlan, PlanItem, TravelPlan, TripRequest

_SLOT_ORDER = ["صباحاً", "ظهراً", "عصراً", "مساءً"]


def build_fallback_plan(request: TripRequest, activities: list[dict], per_day: int = 3) -> TravelPlan:
    """
    خوارزمية جشعة بسيطة:
    - ترتيب الأنشطة حسب مطابقة الاهتمامات ثم الأرخص.
    - توزيعها على الأيام دون تكرار، نشاط واحد لكل فترة زمنية.
    """
    interest = set(request.interest_codes)
    pool = sorted(
        activities,
        key=lambda a: (a["category"] not in interest, a["cost_per_person"]),
    )
    used: set[int] = set()
    days: list[DayPlan] = []

    for day_no in range(1, request.days + 1):
        items: list[PlanItem] = []
        taken_slots: set[str] = set()
        for act in pool:
            if len(items) >= per_day:
                break
            if act["id"] in used or act["best_time"] in taken_slots:
                continue
            items.append(
                PlanItem(
                    activity_id=act["id"],
                    activity_name=act["name"],
                    time_slot=act["best_time"],
                    reason="مطابق لاهتماماتك" if act["category"] in interest else "نشاط متاح مناسب للوقت",
                )
            )
            used.add(act["id"])
            taken_slots.add(act["best_time"])
        if not items:  # نفدت الأنشطة: يوم حر
            break
        items.sort(key=lambda i: _SLOT_ORDER.index(i.time_slot))
        days.append(DayPlan(day_number=day_no, title=f"اليوم {day_no}", items=items))

    return TravelPlan(
        summary=f"خطة مبسطة لـ{request.days} أيام في {request.destination} أُعدّت تلقائياً دون النموذج اللغوي.",
        days=days,
        tips=["هذه خطة احتياطية؛ أعد المحاولة لاحقاً للحصول على خطة مفصلة من المساعد الذكي."],
    )


In [ ]:
# تجربة: خطة بدون نموذج لغوي
from travel_planner.fallback import build_fallback_plan
req = TripRequest(destination="العلا", days=2, travelers=2, budget=6000, interests=["طبيعة ومغامرة"])
fb = build_fallback_plan(req, db.search_activities("العلا", req.interest_codes))
for d in fb.days:
    print(f"اليوم {d.day_number}:", " | ".join(f"{i.time_slot} {i.activity_name}" for i in d.items))

### الخطوة 17: التصدير والاستيراد (`exporter.py`)
تصدير الخطة بصيغة JSON واستيرادها، مع رفض الملفات غير المدعومة أو التالفة.

In [ ]:
%%writefile src/travel_planner/exporter.py
"""
تصدير الخطط واستيرادها (JSON)، مع رفض الملفات غير المدعومة.
"""
from __future__ import annotations

import json

from pydantic import ValidationError

from .exceptions import UnsupportedFileError
from .schemas import TravelPlan

ALLOWED_EXTENSIONS = {".json"}
MAX_FILE_BYTES = 200_000
EXPORT_VERSION = 1


def export_json(request: dict, plan: TravelPlan) -> str:
    payload = {
        "app": "travel-planner",
        "version": EXPORT_VERSION,
        "request": request,
        "plan": plan.model_dump(),
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


def parse_import(filename: str, data: bytes) -> tuple[dict, TravelPlan]:
    """يتحقق من الامتداد والحجم والترميز والبنية، ثم يرجع (الطلب، الخطة)."""
    ext = ("." + filename.rsplit(".", 1)[-1].lower()) if "." in filename else ""
    if ext not in ALLOWED_EXTENSIONS:
        raise UnsupportedFileError(f"الامتداد «{ext or 'بدون امتداد'}» غير مدعوم. يُقبل ملف .json فقط.")
    if len(data) > MAX_FILE_BYTES:
        raise UnsupportedFileError("حجم الملف كبير جداً (الحد 200 كيلوبايت).")
    try:
        payload = json.loads(data.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise UnsupportedFileError("الملف ليس JSON صالحاً أو ترميزه ليس UTF-8.") from exc
    if not isinstance(payload, dict) or payload.get("app") != "travel-planner":
        raise UnsupportedFileError("هذا الملف لم يُصدَّر من مساعد تخطيط السفر.")
    try:
        plan = TravelPlan.model_validate(payload["plan"])
        request = dict(payload["request"])
    except (KeyError, ValidationError, TypeError) as exc:
        raise UnsupportedFileError("بنية الخطة داخل الملف غير صحيحة.") from exc
    return request, plan


In [ ]:
# تجربة: تصدير ثم استيراد، ورفض ملف PDF
from travel_planner.exporter import export_json, parse_import
data = export_json(req.model_dump(), fb)
r, p = parse_import("plan.json", data.encode())
print("✅ استيراد ناجح:", r["destination"], "-", len(p.days), "أيام")
try:
    parse_import("plan.pdf", b"%PDF")
except TravelPlannerError as e:
    print("🛑", e.user_message)

## المرحلة 6: الخدمة الرئيسية
### الخطوة 18: خدمة التخطيط (`planner.py`)
تربط كل الأجزاء: تحقق ← استرجاع ← سلسلة LangChain ← تحقق من الخطة (حتى 3 محاولات) ← حساب الميزانية ← حفظ.

In [ ]:
%%writefile src/travel_planner/planner.py
"""
خدمة التخطيط: الطبقة التي تنسّق بين الواجهة وقاعدة البيانات وسلاسل LangChain.

تدفق إنشاء خطة:
    1) التحقق من المدخلات (validation)
    2) استرجاع الأنشطة (أداة search_activities)
    3) سلسلة LangChain ─► TravelPlan منظم
    4) التحقق من الخطة مقابل البيانات؛ عند الخطأ إعادة المحاولة مع تغذية راجعة
    5) حساب الميزانية برمجياً (أداة calculate_budget)؛ عند التجاوز الكبير محاولة خطة أوفر
    6) الحفظ (أداة save_plan)
    عند تعذّر النموذج: خطة احتياطية بالقواعد مع تنبيه المستخدم.
"""
from __future__ import annotations

import json
import logging
from dataclasses import dataclass, field

from .budget import base_cost, calculate_budget
from .config import Settings, get_settings
from .database import Database
from .exceptions import APIUnavailableError, InvalidInputError, InvalidModelResponseError, TravelPlannerError
from .exporter import parse_import
from .fallback import build_fallback_plan
from .formatting import activities_table, plan_to_text
from .schemas import BudgetReport, TravelPlan, TripRequest
from .validation import validate_plan, validate_request

logger = logging.getLogger(__name__)

MAX_ATTEMPTS = 3          # محاولات توليد خطة صالحة
OVER_BUDGET_TOLERANCE = 0.10  # نسمح بتجاوز 10% قبل طلب خطة أوفر


@dataclass
class PlanResult:
    """ناتج إنشاء أو تعديل خطة، كما تعرضه الواجهة."""

    trip_id: int
    request: dict
    plan: TravelPlan
    budget: BudgetReport
    source: str                       # llm | fallback | imported
    warnings: list[str] = field(default_factory=list)
    attempts: int = 1


class TravelPlannerService:
    """واجهة برمجية واحدة يستخدمها تطبيق Streamlit ودفتر Colab."""

    def __init__(self, settings: Settings | None = None, db: Database | None = None, llm=None):
        self.settings = settings or get_settings()
        self.db = db or Database(self.settings.db_path)
        self.db.initialize()
        self._llm = llm            # يمكن حقن نموذج وهمي في الاختبارات
        self._tools = None
        self._planning_chain = None
        self._revision_chain = None

    # ---------------------------------------------------------- مكونات كسولة
    @property
    def tools(self):
        if self._tools is None:
            from .tools import make_tools  # استيراد متأخر: LangChain مطلوبة هنا فقط
            self._tools = make_tools(self.db)
        return self._tools

    def _get_llm(self):
        if self._llm is None:
            from .llm import build_llm
            self._llm = build_llm(self.settings)
        return self._llm

    def _get_planning_chain(self):
        if self._planning_chain is None:
            llm = self._get_llm()  # يرفع APIUnavailableError أولاً إن لم يوجد مفتاح
            from .chains import build_planning_chain
            self._planning_chain = build_planning_chain(llm, self.tools["search_activities"])
        return self._planning_chain

    def _get_revision_chain(self):
        if self._revision_chain is None:
            llm = self._get_llm()
            from .chains import build_revision_chain
            self._revision_chain = build_revision_chain(llm, self.db)
        return self._revision_chain

    @property
    def llm_available(self) -> bool:
        return self._llm is not None or self.settings.has_api_key

    # ---------------------------------------------------------- إنشاء خطة
    def create_plan(self, raw_request: dict) -> PlanResult:
        request, warnings = validate_request(dict(raw_request), self.db, self.settings)
        destination = self.db.get_destination(request.destination)
        activities = self.db.search_activities(request.destination, request.interest_codes)
        available = {a["id"]: a for a in activities}

        try:
            plan, attempts, extra = self._generate_with_llm(request, destination, available)
            source = "llm"
            warnings += extra
        except (APIUnavailableError, InvalidModelResponseError) as exc:
            logger.warning("Fallback planner used: %s", exc)
            plan = build_fallback_plan(request, activities)
            source, attempts = "fallback", 0
            warnings.append(f"{exc.user_message} — عُرضت خطة احتياطية مبسطة من البيانات المحلية.")

        budget = self._budget(request, plan)
        trip_id = self._save(request, plan, budget.total, source)
        return PlanResult(trip_id, request.model_dump(), plan, budget, source, warnings + budget.warnings, attempts)

    def _generate_with_llm(self, request: TripRequest, destination: dict,
                           available: dict[int, dict]) -> tuple[TravelPlan, int, list[str]]:
        """يستدعي السلسلة حتى MAX_ATTEMPTS مع تغذية راجعة بالأخطاء."""
        chain = self._get_planning_chain()
        base = base_cost(destination, request.days, request.travelers)
        inputs = {
            "destination": request.destination,
            "destination_description": destination["description"],
            "categories": request.interest_codes,
            "days": request.days,
            "travelers": request.travelers,
            "budget": f"{request.budget:,.0f}",
            "base_cost": f"{base:,.0f}",
            "activities_budget": f"{max(request.budget - base, 0):,.0f}",
            "interests": "، ".join(request.interests),
            "notes": request.notes or "لا يوجد",
            "max_per_day": self.settings.max_activities_per_day,
            "feedback": "",
        }
        last_error = ""
        best: TravelPlan | None = None
        for attempt in range(1, MAX_ATTEMPTS + 1):
            try:
                plan: TravelPlan = chain.invoke(inputs)
            except TravelPlannerError:
                raise
            except Exception as exc:  # أخطاء التحليل أو الخدمة
                if _is_parse_error(exc):
                    last_error = "الرد لم يكن JSON مطابقاً للتنسيق."
                    inputs["feedback"] = _feedback([last_error])
                    continue
                from .llm import translate_llm_error
                raise translate_llm_error(exc) from exc

            issues = validate_plan(plan, request, available, self.settings.max_activities_per_day)
            if issues:
                last_error = "؛ ".join(issues)
                inputs["feedback"] = _feedback(issues)
                continue

            best = plan
            report = self._budget(request, plan)
            over = -report.remaining
            if over > request.budget * OVER_BUDGET_TOLERANCE and attempt < MAX_ATTEMPTS:
                inputs["feedback"] = _feedback(
                    [f"الخطة تتجاوز الميزانية بنحو {over:,.0f} ريال؛ استبدل الأنشطة المكلفة بأنشطة مجانية أو أرخص."]
                )
                continue
            return plan, attempt, []

        if best is not None:  # خطة صحيحة لكنها فوق الميزانية بعد كل المحاولات
            return best, MAX_ATTEMPTS, ["تعذّر الوصول لخطة ضمن الميزانية بالأنشطة المتاحة."]
        raise InvalidModelResponseError(f"رد النموذج غير مناسب بعد {MAX_ATTEMPTS} محاولات: {last_error}")

    # ---------------------------------------------------------- تعديل خطة
    def revise_plan(self, trip_id: int, user_request: str) -> PlanResult:
        user_request = (user_request or "").strip()
        if len(user_request) < 3:
            raise InvalidInputError("اكتب طلب التعديل بوضوح (مثال: استبدل نشاط اليوم الثاني بنشاط بحري).")
        if len(user_request) > 400:
            raise InvalidInputError("طلب التعديل طويل جداً (400 حرف كحد أقصى).")

        trip = self.db.get_trip(trip_id)
        if not trip or not trip["plan"]:
            raise InvalidInputError("الرحلة غير موجودة.")
        if not self.llm_available:
            raise APIUnavailableError("التعديل بالمحادثة يحتاج خدمة النموذج اللغوي. أضف مفتاح GROQ_API_KEY.")

        request = TripRequest(**{k: trip[k] for k in ("destination", "days", "travelers", "budget", "interests")})
        activities = self.db.search_activities(request.destination, request.interest_codes)
        available = {a["id"]: a for a in activities}
        chain = self._get_revision_chain()
        inputs = {
            "trip_context": (f"الوجهة {request.destination}، {request.days} أيام، {request.travelers} مسافر، "
                             f"الميزانية {request.budget:,.0f} ريال، الاهتمامات: {'، '.join(request.interests)}"),
            "current_plan": json.dumps(trip["plan"], ensure_ascii=False),
            "activities_table": activities_table(activities),
            "request": user_request,
            "max_per_day": self.settings.max_activities_per_day,
            "feedback": "",
        }
        config = {"configurable": {"session_id": f"trip-{trip_id}"}}

        last_error = ""
        for attempt in range(1, MAX_ATTEMPTS + 1):
            try:
                plan: TravelPlan = chain.invoke(inputs, config=config)
            except TravelPlannerError:
                raise
            except Exception as exc:
                if _is_parse_error(exc):
                    last_error = "الرد لم يكن JSON مطابقاً للتنسيق."
                    inputs["feedback"] = _feedback([last_error])
                    continue
                from .llm import translate_llm_error
                raise translate_llm_error(exc) from exc
            issues = validate_plan(plan, request, available, self.settings.max_activities_per_day)
            if not issues:
                budget = self._budget(request, plan)
                self._save(request, plan, budget.total, "llm", trip_id=trip_id)
                return PlanResult(trip_id, request.model_dump(), plan, budget, "llm", budget.warnings, attempt)
            last_error = "؛ ".join(issues)
            inputs["feedback"] = _feedback(issues)
        raise InvalidModelResponseError(f"تعذّر تطبيق التعديل؛ بقيت الخطة السابقة كما هي. ({last_error})")

    # ---------------------------------------------------------- أدوات مساعدة
    def _budget(self, request: TripRequest, plan: TravelPlan) -> BudgetReport:
        ids = [i.activity_id for d in plan.days for i in d.items]
        if self._tools is not None or _langchain_installed():
            data = self.tools["calculate_budget"].invoke({
                "destination": request.destination, "days": request.days,
                "travelers": request.travelers, "budget": request.budget, "activity_ids": ids,
            })
            return BudgetReport.model_validate(data)
        # مسار بديل إن لم تكن LangChain مثبتة (للاختبارات الأساسية فقط)
        dest = self.db.get_destination(request.destination)
        return calculate_budget(dest, request.days, request.travelers, request.budget,
                                list(self.db.get_activities_by_ids(ids).values()))

    def _save(self, request: TripRequest, plan: TravelPlan, total: float, source: str,
              trip_id: int | None = None) -> int:
        payload = {"request": request.model_dump(), "plan": plan.model_dump(), "total_cost": total,
                   "source": source, "trip_id": trip_id}
        if self._tools is not None or _langchain_installed():
            return self.tools["save_plan"].invoke(payload)
        return self.db.save_trip(payload["request"], payload["plan"], total, source=source, trip_id=trip_id)

    # ---------------------------------------------------------- الخطط المحفوظة
    def list_trips(self) -> list[dict]:
        return self.db.list_trips()

    def load_trip(self, trip_id: int) -> PlanResult:
        trip = self.db.get_trip(trip_id)
        if not trip:
            raise InvalidInputError("الرحلة غير موجودة.")
        request = TripRequest(**{k: trip[k] for k in ("destination", "days", "travelers", "budget", "interests")})
        plan = TravelPlan.model_validate(trip["plan"])
        budget = self._budget(request, plan)
        return PlanResult(trip_id, request.model_dump(), plan, budget, trip["source"], budget.warnings, 0)

    def delete_trip(self, trip_id: int) -> None:
        self.db.delete_trip(trip_id)

    def chat_history(self, trip_id: int) -> list[dict]:
        return self.db.get_messages(f"trip-{trip_id}")

    def export_text(self, result: PlanResult) -> str:
        ids = [i.activity_id for d in result.plan.days for i in d.items]
        return plan_to_text(result.plan, result.request, result.budget, self.db.get_activities_by_ids(ids))

    def import_plan(self, filename: str, data: bytes) -> PlanResult:
        """يستورد خطة JSON ويتحقق منها مقابل قاعدة البيانات قبل حفظها."""
        raw_request, plan = parse_import(filename, data)
        request, _ = validate_request(raw_request, self.db, self.settings)
        available = {a["id"]: a for a in self.db.search_activities(request.destination)}
        issues = validate_plan(plan, request, available, self.settings.max_activities_per_day)
        if issues:
            raise InvalidInputError("الخطة المستوردة لا تطابق البيانات: " + "؛ ".join(issues))
        budget = self._budget(request, plan)
        trip_id = self._save(request, plan, budget.total, "imported")
        return PlanResult(trip_id, request.model_dump(), plan, budget, "imported", budget.warnings, 0)


def _feedback(issues: list[str]) -> str:
    bullet = "\n".join(f"- {i}" for i in issues)
    return f"## تنبيه: محاولتك السابقة رُفضت للأسباب التالية، صحّحها\n{bullet}"


def _is_parse_error(exc: Exception) -> bool:
    return type(exc).__name__ in {"OutputParserException", "ValidationError", "JSONDecodeError"}


def _langchain_installed() -> bool:
    try:
        import langchain_core  # noqa: F401
        return True
    except ImportError:
        return False


In [ ]:
# تجربة حية: إنشاء خطة بالنموذج اللغوي
from travel_planner.planner import TravelPlannerService
service = TravelPlannerService()
print("النموذج:", service.settings.model_name, "| متصل:", service.llm_available)
result = service.create_plan({
    "destination": "المدينة المنورة", "days": 3, "travelers": 2, "budget": 6000,
    "interests": ["تاريخ وتراث", "مطاعم ومقاهي"], "notes": "نفضّل بداية هادئة صباحاً",
})
print("المصدر:", result.source, "| المحاولات:", result.attempts, "| رقم الرحلة:", result.trip_id)
print(service.export_text(result))

### الخطوة 19: تعديل الخطة بالمحادثة (الذاكرة)
طلبان متتاليان؛ الثاني يعتمد على سياق الأول المحفوظ في جدول `chat_messages`.

In [ ]:
r1 = service.revise_plan(result.trip_id, "استبدل نشاط التسوق إن وجد بنشاط تراثي، واجعل اليوم الأول أخف")
print(service.export_text(r1)[:1200])
r2 = service.revise_plan(result.trip_id, "ممتاز، الآن اجعل الخطة أوفر قدر الإمكان دون إلغاء ما عدّلته سابقاً")
print("الإجمالي قبل:", result.budget.total, "| بعد التعديلين:", r2.budget.total)
for m in service.chat_history(result.trip_id):
    print(f"[{m['role']}] {m['content'][:150]}")

### الخطوة 20: حالات الخطأ
نتأكد أن النظام يتعامل بوضوح مع المدخلات الخاطئة والملفات غير المدعومة وانقطاع الخدمة.

In [ ]:
from dataclasses import replace

def attempt(title, fn):
    try:
        out = fn()
        print(f"✅ {title}: نجح ←", getattr(out, "source", out))
    except TravelPlannerError as e:
        print(f"🛑 {title}: {e.user_message}")

base = {"destination": "جدة", "days": 2, "travelers": 2, "budget": 3000, "interests": ["تسوق"]}
attempt("أيام خارج النطاق", lambda: service.create_plan({**base, "days": 12}))
attempt("ميزانية سالبة", lambda: service.create_plan({**base, "budget": -10}))
attempt("وجهة غير متوفرة", lambda: service.create_plan({**base, "destination": "باريس"}))
attempt("ملف غير مدعوم", lambda: service.import_plan("plan.pdf", b"%PDF"))
attempt("طلب تعديل فارغ", lambda: service.revise_plan(result.trip_id, ""))

offline = TravelPlannerService(settings=replace(service.settings, groq_api_key=None))
r = offline.create_plan({**base, "budget": 800})
print("\nبدون مفتاح API ← المصدر:", r.source)
for w in r.warnings:
    print("   ⚠️", w)

In [ ]:
# الخطط المحفوظة في قاعدة البيانات
display(pd.DataFrame(service.list_trips()))

## المرحلة 7: الاختبارات الآلية
### الخطوة 21: ملفات الاختبار وتشغيلها
اختبارات `pytest` تغطي قاعدة البيانات والتحقق والميزانية والخطة الاحتياطية والاستيراد، ومكونات LangChain بنموذج وهمي (بدون استهلاك API).

In [ ]:
%%writefile pytest.ini
[pytest]
testpaths = tests
pythonpath = src


In [ ]:
%%writefile tests/conftest.py
"""إعدادات مشتركة للاختبارات: قاعدة بيانات مؤقتة وإعدادات بلا مفتاح API."""
import sys
from dataclasses import replace
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from travel_planner.config import get_settings  # noqa: E402
from travel_planner.database import Database  # noqa: E402


@pytest.fixture
def settings(tmp_path):
    return replace(get_settings(), groq_api_key=None, db_path=tmp_path / "test.db")


@pytest.fixture
def db(settings):
    database = Database(settings.db_path)
    database.initialize()
    return database


@pytest.fixture
def valid_request():
    return {"destination": "جدة", "days": 2, "travelers": 2, "budget": 5000,
            "interests": ["تاريخ وتراث"], "notes": ""}


In [ ]:
%%writefile tests/test_core.py
"""
اختبارات المنطق الأساسي (لا تحتاج إنترنت ولا LangChain):
قاعدة البيانات، التحقق، الميزانية، الخطة الاحتياطية، الاستيراد والتصدير، معالجة الأخطاء.
"""
import json

import pytest

from travel_planner.budget import calculate_budget, nights_for, rooms_needed
from travel_planner.database import Database
from travel_planner.exceptions import (APIUnavailableError, DatabaseUnavailableError, InvalidInputError,
                                       UnsupportedFileError)
from travel_planner.exporter import export_json, parse_import
from travel_planner.fallback import build_fallback_plan
from travel_planner.planner import TravelPlannerService
from travel_planner.schemas import DayPlan, PlanItem, TravelPlan, TripRequest
from travel_planner.validation import validate_plan, validate_request


# ---------------------------------------------------------------- قاعدة البيانات
def test_seed_creates_three_destinations(db):
    names = [d["name"] for d in db.list_destinations()]
    assert names == ["جدة", "العلا", "المدينة المنورة"]


def test_seed_runs_only_once(db):
    db.initialize()  # تشغيل ثانٍ لا يكرر البيانات
    assert len(db.list_destinations()) == 3


def test_search_prioritises_interests(db):
    acts = db.search_activities("جدة", ["nature"])
    assert acts[0]["category"] == "nature"
    assert all(a["destination_id"] == acts[0]["destination_id"] for a in acts)


def test_unavailable_database_raises_friendly_error(tmp_path):
    blocker = tmp_path / "file.txt"
    blocker.write_text("x")
    bad = Database(blocker / "sub" / "db.sqlite")  # مسار تحت ملف عادي = غير صالح
    with pytest.raises(DatabaseUnavailableError):
        bad.initialize()


# ---------------------------------------------------------------- التحقق من المدخلات
def test_valid_request_passes(db, settings, valid_request):
    req, warnings = validate_request(valid_request, db, settings)
    assert req.days == 2 and warnings == []


@pytest.mark.parametrize("field,value,msg", [
    ("days", 0, "عدد الأيام"),
    ("days", 15, "عدد الأيام"),
    ("travelers", 0, "عدد المسافرين"),
    ("budget", -50, "الميزانية"),
    ("days", "ثلاثة", "رقماً"),
    ("interests", [], "الاهتمامات"),
    ("interests", ["سباحة فضائية"], "غير مدعومة"),
])
def test_invalid_inputs_rejected(db, settings, valid_request, field, value, msg):
    valid_request[field] = value
    with pytest.raises(InvalidInputError) as err:
        validate_request(valid_request, db, settings)
    assert msg in err.value.user_message


def test_unknown_destination_lists_available(db, settings, valid_request):
    valid_request["destination"] = "باريس"
    with pytest.raises(InvalidInputError) as err:
        validate_request(valid_request, db, settings)
    assert "المدينة المنورة" in err.value.user_message


def test_low_budget_gives_warning_not_error(db, settings, valid_request):
    valid_request["budget"] = 300
    _, warnings = validate_request(valid_request, db, settings)
    assert warnings and "أقل من التكلفة الأساسية" in warnings[0]


# ---------------------------------------------------------------- الميزانية
def test_rooms_and_nights():
    assert rooms_needed(1) == 1 and rooms_needed(3) == 2
    assert nights_for(1) == 0 and nights_for(3) == 2


def test_budget_arithmetic(db):
    dest = db.get_destination("المدينة المنورة")  # فندق 450، أكل 120، تنقل 150
    acts = [{"name": "أ", "cost_per_person": 100}, {"name": "ب", "cost_per_person": 0}]
    r = calculate_budget(dest, days=3, travelers=2, budget=5000, activities=acts)
    # سكن 450×1×2=900، أكل 120×2×3=720، تنقل 150×3=450، أنشطة 100×2=200
    assert r.total == 2270
    assert r.remaining == 2730 and r.within_budget


def test_budget_overrun_warning(db):
    dest = db.get_destination("العلا")
    r = calculate_budget(dest, days=3, travelers=4, budget=1000, activities=[])
    assert not r.within_budget
    assert "تتجاوز الميزانية" in r.warnings[0]


# ---------------------------------------------------------------- التحقق من خطة النموذج
def _plan(days):
    return TravelPlan(summary="s", days=[
        DayPlan(day_number=n, title="t", items=[PlanItem(activity_id=i, activity_name="x", time_slot="صباحاً", reason="r")
                                                for i in ids])
        for n, ids in enumerate(days, start=1)
    ])


def test_validate_plan_detects_hallucination_and_duplicates(db):
    req = TripRequest(destination="جدة", days=2, travelers=1, budget=3000, interests=["تسوق"])
    available = {a["id"]: a for a in db.search_activities("جدة")}
    first = next(iter(available))
    issues = validate_plan(_plan([[first, 9999], [first]]), req, available)
    assert any("9999" in i for i in issues)
    assert any("مكرر" in i for i in issues)


def test_validate_plan_wrong_day_count(db):
    req = TripRequest(destination="جدة", days=3, travelers=1, budget=3000, interests=["تسوق"])
    available = {a["id"]: a for a in db.search_activities("جدة")}
    ids = list(available)
    issues = validate_plan(_plan([[ids[0]], [ids[1]]]), req, available)
    assert any("عدد الأيام" in i for i in issues)


def test_validate_plan_fixes_names(db):
    req = TripRequest(destination="جدة", days=1, travelers=1, budget=3000, interests=["تسوق"])
    available = {a["id"]: a for a in db.search_activities("جدة")}
    plan = _plan([[list(available)[0]]])
    assert validate_plan(plan, req, available) == []
    assert plan.days[0].items[0].activity_name == available[list(available)[0]]["name"]


# ---------------------------------------------------------------- الخطة الاحتياطية
def test_fallback_plan_is_valid(db):
    req = TripRequest(destination="العلا", days=3, travelers=2, budget=8000, interests=["طبيعة ومغامرة"])
    acts = db.search_activities("العلا", req.interest_codes)
    plan = build_fallback_plan(req, acts)
    assert len(plan.days) == 3
    assert validate_plan(plan, req, {a["id"]: a for a in acts}) == []


def test_service_without_api_key_uses_fallback(settings, valid_request):
    service = TravelPlannerService(settings=settings)
    result = service.create_plan(valid_request)
    assert result.source == "fallback"
    assert len(result.plan.days) == 2
    assert any("احتياطية" in w for w in result.warnings)
    assert service.list_trips()[0]["id"] == result.trip_id


def test_revision_without_api_key_is_explained(settings, valid_request):
    service = TravelPlannerService(settings=settings)
    trip = service.create_plan(valid_request)
    with pytest.raises(APIUnavailableError):
        service.revise_plan(trip.trip_id, "اجعل اليوم الأول أخف")


# ---------------------------------------------------------------- الاستيراد والتصدير
def test_export_import_roundtrip(settings, valid_request):
    service = TravelPlannerService(settings=settings)
    res = service.create_plan(valid_request)
    data = export_json(res.request, res.plan).encode("utf-8")
    imported = service.import_plan("trip.json", data)
    assert imported.source == "imported" and imported.budget.total == res.budget.total
    assert "اليوم 1" in service.export_text(res)


@pytest.mark.parametrize("name,data,msg", [
    ("plan.pdf", b"%PDF-1.4", "غير مدعوم"),
    ("plan.json", b"not json", "ليس JSON"),
    ("plan.json", json.dumps({"hello": 1}).encode(), "لم يُصدَّر"),
    ("plan.json", json.dumps({"app": "travel-planner", "request": {}, "plan": {"x": 1}}).encode(), "غير صحيحة"),
])
def test_unsupported_files_rejected(name, data, msg):
    with pytest.raises(UnsupportedFileError) as err:
        parse_import(name, data)
    assert msg in err.value.user_message


In [ ]:
%%writefile tests/test_langchain.py
"""
اختبارات مكونات LangChain بنموذج وهمي (FakeListChatModel) — لا تحتاج مفتاح API.
تُتخطى تلقائياً إذا لم تكن LangChain مثبتة. تعمل في Colab بعد تثبيت requirements.txt.
"""
import json

import pytest

pytest.importorskip("langchain_core")

from langchain_core.language_models.fake_chat_models import FakeListChatModel  # noqa: E402

from travel_planner.chains import SQLiteChatHistory  # noqa: E402
from travel_planner.formatting import activities_table  # noqa: E402
from travel_planner.planner import TravelPlannerService  # noqa: E402
from travel_planner.prompts import planning_prompt  # noqa: E402
from travel_planner.tools import make_tools  # noqa: E402


def plan_json(days_ids, summary="رحلة تراثية"):
    return json.dumps({
        "summary": summary,
        "days": [
            {"day_number": n, "title": f"يوم {n}",
             "items": [{"activity_id": i, "activity_name": "اسم", "time_slot": "صباحاً", "reason": "مناسب"} for i in ids]}
            for n, ids in enumerate(days_ids, start=1)
        ],
        "tips": ["احجز مبكراً"],
    }, ensure_ascii=False)


def service_with(settings, responses):
    return TravelPlannerService(settings=settings, llm=FakeListChatModel(responses=responses))


def test_prompt_contains_rules_and_table(db):
    acts = db.search_activities("جدة")
    msgs = planning_prompt().format_messages(
        destination="جدة", destination_description="", days=2, travelers=2, budget="5,000",
        base_cost="1,000", activities_budget="4,000", interests="تسوق", notes="", max_per_day=4,
        activities_table=activities_table(acts), feedback="", format_instructions="JSON")
    assert "لا تخترع" in msgs[0].content
    assert acts[0]["name"] in msgs[1].content


def test_tools_work(db):
    tools = make_tools(db)
    acts = tools["search_activities"].invoke({"destination": "جدة", "categories": ["food"]})
    assert acts[0]["category"] == "food"
    report = tools["calculate_budget"].invoke({"destination": "جدة", "days": 1, "travelers": 1,
                                                "budget": 1000, "activity_ids": [acts[0]["id"]]})
    assert report["total"] > 0


def test_chain_produces_structured_plan(settings, valid_request):
    service = service_with(settings, [f"```json\n{plan_json([[1, 2], [3, 4]])}\n```"])
    result = service.create_plan(valid_request)
    assert result.source == "llm" and result.attempts == 1
    assert len(result.plan.days) == 2
    # الاسم صُحّح من قاعدة البيانات بدل "اسم"
    assert result.plan.days[0].items[0].activity_name != "اسم"


def test_hallucinated_activity_triggers_retry(settings, valid_request):
    service = service_with(settings, [plan_json([[999, 2], [3, 4]]), plan_json([[1, 2], [3, 4]])])
    result = service.create_plan(valid_request)
    assert result.source == "llm" and result.attempts == 2


def test_malformed_response_triggers_retry(settings, valid_request):
    service = service_with(settings, ["عذراً لا أستطيع", plan_json([[1, 2], [3, 4]])])
    assert service.create_plan(valid_request).attempts == 2


def test_persistently_bad_model_falls_back(settings, valid_request):
    service = service_with(settings, ["نص غير منظم"])
    result = service.create_plan(valid_request)
    assert result.source == "fallback"
    assert any("غير مناسب" in w for w in result.warnings)


def test_revision_uses_memory(settings, valid_request):
    service = service_with(settings, [plan_json([[1, 2], [3, 4]]),
                                      plan_json([[5, 6], [3, 4]], summary="بعد التعديل")])
    created = service.create_plan(valid_request)
    revised = service.revise_plan(created.trip_id, "غيّر أنشطة اليوم الأول")
    assert revised.plan.summary == "بعد التعديل"
    history = SQLiteChatHistory(service.db, f"trip-{created.trip_id}").messages
    assert [m.type for m in history] == ["human", "ai"]
    assert service.db.get_trip(created.trip_id)["plan"]["summary"] == "بعد التعديل"


In [ ]:
!python -m pytest -q

## المرحلة 8: واجهة المستخدم
### الخطوة 22: لمسات الذكاء الاصطناعي في الواجهة (`ai_theme.py`)
شبكة عصبية متحركة، كرة المساعد، خط تفكير المساعد أثناء بناء الخطة، وبطاقة تحليل بأرقام محسوبة فعلياً.

In [ ]:
%%writefile ai_theme.py
"""
لمسات الذكاء الاصطناعي في الواجهة: شبكة عصبية متحركة، كرة المساعد المتوهجة،
خط تفكير المساعد أثناء بناء الخطة، وبطاقة تحليل الذكاء الاصطناعي بعد الخطة.
كل الأرقام المعروضة محسوبة فعلياً من الخطة، وليست زينة.
"""
from __future__ import annotations

import random

import streamlit as st

AI_CSS = """
<style>
  /* إخفاء الشريط الجانبي المطوي بالكامل (كان يظهر كخط عمودي في الشاشات الضيقة) */
  section[data-testid="stSidebar"] { overflow: hidden; }

  /* الشبكة العصبية خلف البطل */
  .hero .neural { position:absolute; inset:0; width:100%; height:100%; z-index:0; opacity:.55; pointer-events:none; }
  .neural line { stroke:url(#ngrad); stroke-width:1; opacity:.35; }
  .neural circle { fill:#00E5FF; filter:drop-shadow(0 0 4px #00E5FF); animation: pulse 3s ease-in-out infinite; }
  .neural circle:nth-child(3n) { fill:#B79CFF; filter:drop-shadow(0 0 4px #B79CFF); animation-delay:1s; }
  .neural circle:nth-child(4n) { animation-delay:2s; }
  .neural .signal { stroke:#00E5FF; stroke-width:2; stroke-dasharray:6 140; opacity:.9;
                    animation: flow 3.5s linear infinite; }
  @keyframes pulse { 0%,100% { opacity:.35; } 50% { opacity:1; } }
  @keyframes flow { to { stroke-dashoffset:-146; } }

  /* كرة المساعد */
  .hero-row { display:flex; align-items:center; gap:26px; }
  .orb { flex:0 0 auto; width:96px; height:96px; border-radius:50%; position:relative;
         background: radial-gradient(circle at 35% 30%, #fff 0%, #7CF6FF 18%, #00E5FF 38%, #7C4DFF 75%, #2a1a6e 100%);
         box-shadow: 0 0 30px rgba(0,229,255,.7), 0 0 70px rgba(124,77,255,.5); animation: breathe 3.2s ease-in-out infinite; }
  .orb:before, .orb:after { content:""; position:absolute; inset:-12px; border-radius:50%;
         border:1px solid rgba(0,229,255,.55); animation: ring 3.2s ease-out infinite; }
  .orb:after { animation-delay:1.6s; border-color: rgba(183,156,255,.55); }
  @keyframes breathe { 0%,100% { transform:scale(1); } 50% { transform:scale(1.06); } }
  @keyframes ring { from { transform:scale(.9); opacity:1; } to { transform:scale(1.5); opacity:0; } }

  /* سطر الكتابة الآلية */
  .typing { display:inline-block; margin-top:12px; font-weight:700; color:#7CF6FF; white-space:nowrap; overflow:hidden;
            vertical-align:bottom; border-left:2px solid #00E5FF; padding-left:4px; max-width:0;
            animation: type 3.2s steps(40) .4s forwards, caret .8s step-end infinite; }
  @keyframes type { to { max-width: 40em; } }
  @keyframes caret { 50% { border-color: transparent; } }

  /* خط تفكير المساعد */
  .pipeline { display:flex; gap:10px; flex-wrap:wrap; margin: 6px 0 18px; }
  .step { flex:1 1 150px; padding:12px 14px; border-radius:16px; border:1px solid rgba(0,229,255,.25);
          background: rgba(19,28,66,.72); color:#e7ebff; font-weight:700; font-size:.92rem; position:relative; overflow:hidden; }
  .step small { display:block; color:#AEB9D8; font-weight:500; margin-top:4px; }
  .step.run { border-color:#00E5FF; box-shadow:0 0 18px rgba(0,229,255,.35); }
  .step.run:after { content:""; position:absolute; top:0; bottom:0; width:40%; right:-40%;
          background: linear-gradient(90deg, transparent, rgba(0,229,255,.25), transparent); animation: scan 1.4s linear infinite; }
  .step.done { border-color:#48F0B0; }
  .step.done:before { content:"✓ "; color:#48F0B0; }
  @keyframes scan { to { right: 110%; } }

  /* بطاقة تحليل الذكاء الاصطناعي */
  .insight { display:grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap:12px; margin: 6px 0 20px; }
  .ins { padding:14px 16px; border-radius:18px; background: rgba(19,28,66,.72); border:1px solid rgba(183,156,255,.3); }
  .ins .v { font-size:1.7rem; font-weight:800; background: linear-gradient(90deg, #00E5FF, #B79CFF);
            -webkit-background-clip:text; background-clip:text; color:transparent; }
  .ins .l { color:#AEB9D8; font-size:.88rem; }
  .bar { height:6px; border-radius:6px; background: rgba(255,255,255,.08); margin-top:8px; overflow:hidden; }
  .bar i { display:block; height:100%; background: linear-gradient(90deg, #48F0B0, #00E5FF); border-radius:6px; }

  /* حالة النموذج في الشريط الجانبي */
  .ai-status { display:flex; align-items:center; gap:10px; padding:10px 12px; border-radius:14px;
               background: rgba(0,229,255,.06); border:1px solid rgba(0,229,255,.25); font-weight:700; }
  .dot { width:10px; height:10px; border-radius:50%; background:#48F0B0; box-shadow:0 0 10px #48F0B0;
         animation: pulse 1.6s ease-in-out infinite; }
  .dot.off { background:#FFC857; box-shadow:0 0 10px #FFC857; }
</style>
"""

STEPS = [
    ("🧠", "فهم الطلب", "تحليل الوجهة والاهتمامات"),
    ("🔎", "استرجاع الأنشطة", "من قاعدة بيانات SQLite"),
    ("✨", "النموذج يخطط", "GPT-OSS 120B عبر LangChain"),
    ("🛡️", "التحقق", "رفض أي نشاط مختلق"),
    ("🧮", "حساب الميزانية", "برمجياً لا بالنموذج"),
]


def inject_css() -> None:
    st.markdown(AI_CSS, unsafe_allow_html=True)


def _neural_svg(seed: int = 7, n: int = 26) -> str:
    """رسم شبكة عصبية عشوائية ثابتة (نفس الشكل في كل تحميل)."""
    rnd = random.Random(seed)
    pts = [(rnd.randint(10, 990), rnd.randint(10, 290)) for _ in range(n)]
    lines, signals = [], []
    for i, (x1, y1) in enumerate(pts):
        near = sorted(pts, key=lambda p: (p[0] - x1) ** 2 + (p[1] - y1) ** 2)[1:3]
        for (x2, y2) in near:
            lines.append(f"<line x1='{x1}' y1='{y1}' x2='{x2}' y2='{y2}'/>")
            if i % 4 == 0:
                signals.append(f"<line class='signal' x1='{x1}' y1='{y1}' x2='{x2}' y2='{y2}'/>")
    dots = "".join(f"<circle cx='{x}' cy='{y}' r='{rnd.choice([2, 2.5, 3.5])}'/>" for x, y in pts)
    return ("<svg class='neural' viewBox='0 0 1000 300' preserveAspectRatio='xMidYMid slice'>"
            "<defs><linearGradient id='ngrad'><stop offset='0' stop-color='#00E5FF'/>"
            "<stop offset='1' stop-color='#7C4DFF'/></linearGradient></defs>"
            f"{''.join(lines)}{''.join(signals)}{dots}</svg>")


def ai_hero(title: str, subtitle: str, badge: str, typing: str | None = None, chips: str = "") -> None:
    typing_html = f"<div class='typing'>{typing}</div>" if typing else ""
    st.markdown(
        f"<div class='hero'>{_neural_svg()}<div class='hero-row'><div class='orb'></div><div>"
        f"<span class='badge'>{badge}</span><h1>{title}</h1><p>{subtitle}</p>{typing_html}</div></div>"
        f"{chips}</div>",
        unsafe_allow_html=True,
    )


def pipeline_html(active: int | None) -> str:
    """active=None يعني اكتملت كل المراحل."""
    cells = []
    for i, (icon, name, desc) in enumerate(STEPS):
        cls = "done" if active is None or i < active else ("run" if i == active else "")
        cells.append(f"<div class='step {cls}'>{icon} {name}<small>{desc}</small></div>")
    return f"<div class='pipeline'>{''.join(cells)}</div>"


def insight_cards(match_pct: int, n_acts: int, budget_pct: int, attempts: int, source: str) -> None:
    engine = "GPT-OSS 120B · Groq" if source == "llm" else ("مخطط احتياطي" if source == "fallback" else "خطة مستوردة")
    st.markdown(
        "<div class='section-title'>🤖 تحليل المساعد الذكي</div><div class='insight'>"
        f"<div class='ins'><div class='v'>{match_pct}%</div><div class='l'>مطابقة اهتماماتك</div>"
        f"<div class='bar'><i style='width:{match_pct}%'></i></div></div>"
        f"<div class='ins'><div class='v'>{n_acts}</div><div class='l'>نشاطاً موثقاً · 0 مختلق</div></div>"
        f"<div class='ins'><div class='v'>{budget_pct}%</div><div class='l'>من الميزانية مستخدمة</div>"
        f"<div class='bar'><i style='width:{min(budget_pct, 100)}%'></i></div></div>"
        f"<div class='ins'><div class='v'>{attempts or 1}</div><div class='l'>محاولات حتى اجتياز التحقق</div></div>"
        f"<div class='ins'><div class='v' style='font-size:1.1rem'>{engine}</div><div class='l'>محرك التخطيط</div></div>"
        "</div>",
        unsafe_allow_html=True,
    )


def ai_status(online: bool, model: str) -> None:
    label = f"متصل · {model}" if online else "الخطة الاحتياطية"
    st.markdown(f"<div class='ai-status'><span class='dot{'' if online else ' off'}'></span>{label}</div>",
                unsafe_allow_html=True)


### الخطوة 23: واجهة Streamlit (`app.py`)
ثلاث صفحات: تخطيط رحلة، الخطط المحفوظة، عن النظام. بتصميم داكن ودعم كامل للعربية من اليمين لليسار.

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "dark"
primaryColor = "#00E5FF"
backgroundColor = "#070B1F"
secondaryBackgroundColor = "#11183A"
textColor = "#F4F7FF"
font = "sans serif"

[server]
headless = true


In [ ]:
%%writefile app.py
"""
واجهة المستخدم — مساعد تخطيط السفر (Streamlit).

التشغيل:
    streamlit run app.py
"""
from __future__ import annotations

import sys
import time
from pathlib import Path

import streamlit as st

sys.path.insert(0, str(Path(__file__).parent / "src"))

from travel_planner.exceptions import TravelPlannerError  # noqa: E402
from travel_planner.exporter import export_json  # noqa: E402
from travel_planner.formatting import CATEGORY_AR  # noqa: E402
from travel_planner.planner import PlanResult, TravelPlannerService  # noqa: E402
from travel_planner.schemas import SUPPORTED_INTERESTS  # noqa: E402

import ai_theme  # noqa: E402,I001  لمسات الذكاء الاصطناعي في الواجهة

st.set_page_config(page_title="مساعد تخطيط السفر", page_icon="🧳", layout="wide")

# هوية بصرية داكنة متوهجة توحي بالذكاء الاصطناعي، مع دعم كامل للاتجاه من اليمين لليسار
st.markdown(
    """
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700;800&display=swap');

      :root { --cyan:#00E5FF; --violet:#7C4DFF; --lilac:#B79CFF; --mint:#48F0B0;
              --bg:#070B1F; --card:rgba(19,28,66,.72); --line:rgba(0,229,255,.28); --muted:#AEB9D8; }

      html, body, [class*="css"], .stApp, .stMarkdown, .stTextInput, .stSelectbox, .stMultiSelect,
      .stNumberInput, .stSlider, .stChatMessage, .stAlert, .stTabs, .stRadio, .stCaption, label, p, h1, h2, h3, h4 {
          direction: rtl; text-align: right; font-family: 'Tajawal', sans-serif !important; }

      /* خلفية بتدرجات وشبكة نقاط خفيفة */
      .stApp {
          background:
            radial-gradient(1100px 600px at 85% -10%, rgba(124,77,255,.28), transparent 60%),
            radial-gradient(900px 500px at 5% 10%, rgba(0,229,255,.18), transparent 60%),
            radial-gradient(rgba(255,255,255,.05) 1px, transparent 1px) 0 0 / 26px 26px,
            var(--bg);
      }
      header[data-testid="stHeader"] { background: transparent; }

      /* الشريط الجانبي الزجاجي */
      section[data-testid="stSidebar"] > div {
          background: linear-gradient(180deg, rgba(17,24,58,.95), rgba(7,11,31,.95));
          border-left: 1px solid var(--line); }
      section[data-testid="stSidebar"] .stRadio label { font-size: 1.05rem; padding: 6px 2px; }

      /* البطل (Hero) */
      .hero { position: relative; padding: 34px 34px 28px; border-radius: 26px; margin: 4px 0 26px;
              background: linear-gradient(135deg, rgba(0,229,255,.10), rgba(124,77,255,.16));
              border: 1px solid var(--line); box-shadow: 0 0 40px rgba(0,229,255,.12), inset 0 0 60px rgba(124,77,255,.10);
              overflow: hidden; }
      .hero:after { content:""; position:absolute; inset:-40%; background:
              conic-gradient(from 180deg, transparent, rgba(0,229,255,.18), transparent 30%);
              animation: spin 14s linear infinite; z-index:0; opacity:.45; }
      .hero > * { position: relative; z-index: 1; }
      @keyframes spin { to { transform: rotate(360deg); } }
      .hero .badge { display:inline-block; padding:6px 14px; border-radius:999px; font-size:.85rem; font-weight:700;
              color:var(--cyan); border:1px solid var(--line); background:rgba(0,229,255,.08); margin-bottom:12px; }
      .hero h1 { margin:0; font-size: 2.6rem; font-weight: 800; line-height:1.25;
              background: linear-gradient(90deg, #fff, var(--cyan) 45%, var(--lilac));
              -webkit-background-clip: text; background-clip: text; color: transparent; }
      .hero p { color: var(--muted); font-size: 1.08rem; margin: 10px 0 0; }
      .chips { display:flex; gap:10px; flex-wrap:wrap; margin-top:18px; }
      .chip { padding:8px 14px; border-radius:14px; background:rgba(7,11,31,.55); border:1px solid rgba(183,156,255,.35);
              color:#e7ebff; font-size:.92rem; }
      .chip b { color: var(--cyan); }

      /* البطاقات */
      .day-card { border: 1px solid var(--line); border-radius: 20px; padding: 18px 22px; margin-bottom: 14px;
                  background: var(--card); backdrop-filter: blur(6px);
                  box-shadow: 0 0 24px rgba(0,229,255,.08); transition: transform .2s, box-shadow .2s; }
      .day-card:hover { transform: translateY(-2px); box-shadow: 0 0 34px rgba(0,229,255,.22); }
      .day-card h4 { margin: 0 0 10px; color: #fff; }
      .day-card h4 span { color: var(--cyan); }
      .day-card b { color: #fff; }
      .item { padding: 10px 0; border-top: 1px dashed rgba(174,185,216,.18); }
      .item:first-of-type { border-top: 0; }
      .slot { display:inline-block; min-width:70px; padding:2px 10px; margin-left:8px; border-radius:10px; font-weight:700;
              color:#070B1F; background: linear-gradient(90deg, var(--cyan), var(--lilac)); font-size:.85rem; text-align:center; }
      .muted { color: var(--muted); font-size: 0.92rem; }
      .section-title { font-size:1.35rem; font-weight:800; margin: 8px 0 12px; color:#fff; }
      .section-title:before { content:""; display:inline-block; width:10px; height:10px; border-radius:50%;
              background: var(--cyan); box-shadow: 0 0 12px var(--cyan); margin-left:10px; }

      /* عناصر Streamlit */
      div[data-testid="stForm"], div[data-testid="stMetric"], .stTable, div[data-testid="stDataFrame"] {
          background: var(--card); border: 1px solid var(--line); border-radius: 20px; }
      div[data-testid="stForm"] { padding: 22px 24px; box-shadow: 0 0 30px rgba(124,77,255,.12); }
      div[data-testid="stMetric"] { padding: 14px 16px; }
      div[data-testid="stMetricValue"] { color: var(--cyan); font-weight: 800; }
      .stButton > button, .stDownloadButton > button, div[data-testid="stFormSubmitButton"] > button {
          border-radius: 14px; border: 1px solid var(--line); font-weight: 700;
          background: rgba(0,229,255,.06); color: #fff; transition: all .2s; }
      .stButton > button:hover, .stDownloadButton > button:hover { border-color: var(--cyan); box-shadow: 0 0 18px rgba(0,229,255,.35); }
      div[data-testid="stFormSubmitButton"] > button[kind="primaryFormSubmit"], .stButton > button[kind="primary"] {
          background: linear-gradient(90deg, var(--cyan), var(--violet)); color: #070B1F; border: 0;
          box-shadow: 0 0 24px rgba(0,229,255,.35); }
      div[data-testid="stProgress"] > div > div > div { background: linear-gradient(90deg, var(--mint), var(--cyan)); }
      div[data-testid="stChatMessage"] { background: var(--card); border: 1px solid rgba(183,156,255,.25); border-radius: 18px; }
      .stTextInput input, .stNumberInput input, div[data-baseweb="select"] > div {
          background: rgba(7,11,31,.6) !important; border-radius: 12px !important; }
      .footer { text-align:center; color: var(--muted); font-size:.85rem; margin-top: 40px; opacity:.8; }
    </style>
    """,
    unsafe_allow_html=True,
)
ai_theme.inject_css()


def hero(title: str, subtitle: str, badge: str = "🤖 مدعوم بالذكاء الاصطناعي · LangChain + GPT-OSS 120B",
         typing: str | None = None) -> None:
    """يعرض رأس الصفحة المتوهج مع الشبكة العصبية وكرة المساعد."""
    chips = ("<div class='chips'><span class='chip'>📍 <b>3</b> وجهات</span>"
             "<span class='chip'>🎯 <b>33</b> نشاطاً موثقاً</span>"
             "<span class='chip'>🧮 ميزانية تحسب بالكود</span>"
             "<span class='chip'>💬 تعديل بالمحادثة</span></div>")
    ai_theme.ai_hero(title, subtitle, badge, typing=typing, chips=chips)


SOURCE_LABELS = {"llm": "🤖 مولّدة بالنموذج اللغوي", "fallback": "⚙️ خطة احتياطية", "imported": "📥 مستوردة"}


def _load_cloud_secret() -> None:
    """على Streamlit Cloud يوضع المفتاح في Secrets؛ ننقله لمتغيرات البيئة دون كتابته في الكود."""
    import os
    try:
        if "GROQ_API_KEY" in st.secrets and not os.getenv("GROQ_API_KEY"):
            os.environ["GROQ_API_KEY"] = st.secrets["GROQ_API_KEY"]
    except Exception:  # noqa: BLE001 — لا يوجد ملف أسرار محلياً
        pass


@st.cache_resource(show_spinner=False)
def get_service() -> TravelPlannerService:
    return TravelPlannerService()


def show_error(exc: Exception) -> None:
    """يعرض الأخطاء المعروفة برسالة عربية، وغير المعروفة برسالة عامة."""
    if isinstance(exc, TravelPlannerError):
        st.error(exc.user_message)
    else:
        st.error("حدث خطأ غير متوقع. أعد المحاولة، وإن تكرر راجع سجل التشغيل.")
        st.exception(exc)


def render_plan(service: TravelPlannerService, result: PlanResult) -> None:
    req = result.request
    st.markdown(f"<div class='section-title'>رحلتك إلى {req['destination']} لمدة {req['days']} أيام</div>",
                unsafe_allow_html=True)
    st.caption(f"رقم الرحلة #{result.trip_id} · {SOURCE_LABELS.get(result.source, result.source)}"
               + (f" · عدد المحاولات {result.attempts}" if result.attempts else ""))
    st.write(result.plan.summary)

    for w in result.warnings:
        st.warning(w)

    ids = [i.activity_id for d in result.plan.days for i in d.items]
    acts = service.db.get_activities_by_ids(ids)

    # بطاقة تحليل المساعد: أرقام محسوبة فعلياً من الخطة
    codes = {SUPPORTED_INTERESTS[i] for i in req.get("interests", []) if i in SUPPORTED_INTERESTS}
    matched = sum(1 for a in acts.values() if a.get("category") in codes)
    match_pct = round(100 * matched / len(acts)) if acts and codes else 100
    budget_pct = round(100 * result.budget.total / result.budget.budget) if result.budget.budget else 0
    ai_theme.insight_cards(match_pct, len(ids), budget_pct, result.attempts, result.source)

    col_plan, col_budget = st.columns([3, 2], gap="large")

    with col_plan:
        for day in result.plan.days:
            items_html = "".join(
                f"<div class='item'><span class='slot'>{it.time_slot}</span> <b>{it.activity_name}</b>"
                f" <span class='muted'>· {CATEGORY_AR.get(acts.get(it.activity_id, {}).get('category'), '')}"
                f" · {acts.get(it.activity_id, {}).get('cost_per_person', 0):g} ريال/فرد</span>"
                f"<div class='muted'>{it.reason}</div></div>"
                for it in day.items
            )
            st.markdown(f"<div class='day-card'><h4><span>اليوم {day.day_number}</span> · {day.title}</h4>{items_html}</div>",
                        unsafe_allow_html=True)
        if result.plan.tips:
            st.markdown("**نصائح:**\n" + "\n".join(f"- {t}" for t in result.plan.tips))

    with col_budget:
        b = result.budget
        st.markdown("<div class='section-title'>التكلفة التقديرية</div>", unsafe_allow_html=True)
        st.table({"البند": [line.label for line in b.lines], "المبلغ (ريال)": [f"{line.amount:,.0f}" for line in b.lines]})
        m1, m2 = st.columns(2)
        m1.metric("الإجمالي", f"{b.total:,.0f} ريال")
        m2.metric("المتبقي", f"{b.remaining:,.0f} ريال", delta=None if b.within_budget else "تجاوز")
        st.progress(min(b.total / b.budget, 1.0), text=f"{b.total / b.budget:.0%} من الميزانية")
        st.caption("التكاليف تقديرية من بيانات تجريبية، ويحسبها النظام برمجياً لا النموذج.")

        st.download_button("⬇️ تصدير نصي (.txt)", service.export_text(result),
                           file_name=f"trip_{result.trip_id}.txt", use_container_width=True)
        st.download_button("⬇️ تصدير JSON", export_json(result.request, result.plan),
                           file_name=f"trip_{result.trip_id}.json", use_container_width=True)


def render_chat(service: TravelPlannerService, result: PlanResult) -> None:
    st.divider()
    st.markdown("<div class='section-title'>💬 عدّل خطتك بالمحادثة</div>", unsafe_allow_html=True)
    st.caption("أمثلة: «اجعل اليوم الأول أخف» · «استبدل التسوق بنشاط تراثي» · «خطة أوفر»")
    for msg in service.chat_history(result.trip_id):
        is_user = msg["role"] == "human"
        with st.chat_message("user" if is_user else "assistant", avatar="🧑" if is_user else "✨"):
            st.write(msg["content"])

    prompt = st.chat_input("اكتب طلب التعديل...")
    if prompt:
        with st.spinner("✨ المساعد الذكي يعيد التفكير في خطتك..."):
            try:
                st.session_state.result = service.revise_plan(result.trip_id, prompt)
                st.rerun()
            except Exception as exc:  # noqa: BLE001
                show_error(exc)


def page_plan(service: TravelPlannerService) -> None:
    hero("مساعد تخطيط السفر الذكي",
         "أخبرني بوجهتك وميزانيتك واهتماماتك، وسأبني لك برنامجاً يومياً دقيقاً من أنشطة حقيقية خلال ثوانٍ.",
         typing="أفهم طلبك… أسترجع الأنشطة… أخطط… أتحقق… أحسب")

    if not service.llm_available:
        st.info("لم يُضبط مفتاح GROQ_API_KEY؛ سيعمل التطبيق بالخطة الاحتياطية دون النموذج اللغوي.")

    try:
        destinations = service.db.list_destinations()
    except Exception as exc:  # noqa: BLE001
        show_error(exc)
        return

    with st.form("trip_form"):
        c1, c2, c3 = st.columns(3)
        dest = c1.selectbox("الوجهة", [d["name"] for d in destinations])
        days = c2.number_input("عدد الأيام", min_value=1, max_value=7, value=3)
        travelers = c3.number_input("عدد المسافرين", min_value=1, max_value=10, value=2)
        budget = st.number_input("الميزانية الإجمالية (ريال)", min_value=0, value=5000, step=500)
        interests = st.multiselect("الاهتمامات", list(SUPPORTED_INTERESTS), default=["تاريخ وتراث"])
        notes = st.text_input("ملاحظات (اختياري)", max_chars=300, placeholder="مثال: معنا أطفال، نفضّل بداية متأخرة")
        submitted = st.form_submit_button("✨ أنشئ الخطة", type="primary", use_container_width=True)

    if submitted:
        # خط تفكير المساعد: يعرض مراحل المعالجة الفعلية أثناء بناء الخطة
        board = st.empty()
        for step in range(2):
            board.markdown(ai_theme.pipeline_html(step), unsafe_allow_html=True)
            time.sleep(0.35)
        board.markdown(ai_theme.pipeline_html(2), unsafe_allow_html=True)
        with st.spinner("✨ المساعد الذكي يبني خطتك..."):
            try:
                st.session_state.result = service.create_plan({
                    "destination": dest, "days": days, "travelers": travelers,
                    "budget": budget, "interests": interests, "notes": notes,
                })
                board.markdown(ai_theme.pipeline_html(None), unsafe_allow_html=True)
            except Exception as exc:  # noqa: BLE001
                board.empty()
                show_error(exc)

    result: PlanResult | None = st.session_state.get("result")
    if result:
        render_plan(service, result)
        render_chat(service, result)


def page_saved(service: TravelPlannerService) -> None:
    hero("رحلاتي المحفوظة", "كل خططك في مكان واحد: افتحها، صدرها، أو احذفها.", badge="📁 الخطط المحفوظة")
    try:
        trips = service.list_trips()
    except Exception as exc:  # noqa: BLE001
        show_error(exc)
        return

    up = st.file_uploader("استيراد خطة (ملف JSON صادر من التطبيق)")
    if up is not None and st.button("استيراد"):
        try:
            st.session_state.result = service.import_plan(up.name, up.getvalue())
            st.success("تم الاستيراد والحفظ.")
        except Exception as exc:  # noqa: BLE001
            show_error(exc)

    if not trips:
        st.info("لا توجد خطط محفوظة بعد.")
        return

    st.dataframe(
        [{"#": t["id"], "الوجهة": t["destination"], "الأيام": t["days"], "المسافرون": t["travelers"],
          "الميزانية": t["budget"], "التكلفة": t["total_cost"], "المصدر": SOURCE_LABELS.get(t["source"], t["source"]),
          "آخر تحديث": t["updated_at"]} for t in trips],
        use_container_width=True, hide_index=True,
    )
    selected = st.selectbox("اختر رحلة", [t["id"] for t in trips],
                            format_func=lambda i: f"#{i} — " + next(t["destination"] for t in trips if t["id"] == i))
    c1, c2 = st.columns(2)
    if c1.button("فتح الرحلة", use_container_width=True):
        st.session_state.result = service.load_trip(selected)
        st.session_state.nav_target = "تخطيط رحلة"  # يُطبق قبل إنشاء القائمة في الدورة التالية
        st.rerun()
    if c2.button("🗑️ حذف", use_container_width=True):
        service.delete_trip(selected)
        st.session_state.pop("result", None)
        st.rerun()


def page_about(service: TravelPlannerService) -> None:
    hero("كيف يعمل المساعد؟", "نموذج لغوي مرن، وقاعدة بيانات موثوقة، وحساب برمجي دقيق في نظام واحد.", badge="ℹ️ عن النظام")
    st.markdown(
        f"""
        - **النموذج اللغوي:** `{service.settings.model_name}` عبر Groq ({'متصل' if service.llm_available else 'غير مضبوط'})
        - **التنسيق:** LangChain — ChatPromptTemplate · LCEL · PydanticOutputParser · Tools · RunnableWithMessageHistory
        - **قاعدة البيانات:** SQLite (`{service.settings.db_path.name}`)
        - **الوجهات:** {len(service.db.list_destinations())} وجهات بأنشطة وتكاليف تقديرية تجريبية
        """
    )
    st.subheader("الأنشطة المتاحة")
    dest = st.selectbox("الوجهة", [d["name"] for d in service.db.list_destinations()])
    rows = service.db.search_activities(dest)
    st.dataframe([{"#": a["id"], "النشاط": a["name"], "الفئة": CATEGORY_AR[a["category"]],
                   "المدة": a["duration_hours"], "التكلفة/فرد": a["cost_per_person"], "الوقت": a["best_time"]}
                  for a in rows], hide_index=True, use_container_width=True)


def main() -> None:
    _load_cloud_secret()
    try:
        service = get_service()
    except Exception as exc:  # noqa: BLE001
        show_error(exc)
        st.stop()

    pages = {"تخطيط رحلة": page_plan, "الخطط المحفوظة": page_saved, "عن النظام": page_about}
    if "nav_target" in st.session_state:
        st.session_state.page = st.session_state.pop("nav_target")
    with st.sidebar:
        st.markdown("<div class='section-title'>🧭 مساعد السفر</div>", unsafe_allow_html=True)
        page = st.radio("القائمة", list(pages), key="page", label_visibility="collapsed")
        st.divider()
        ai_theme.ai_status(service.llm_available, "GPT-OSS 120B")
        st.caption("مشروع مقرر أساسيات LangChain · المجموعة 12\nجامعة الأمير مساعد بن عبدالرحمن")
    pages[page](service)
    st.markdown("<div class='footer'>صنع بواسطة المجموعة 12 · مساعد تخطيط السفر الذكي</div>", unsafe_allow_html=True)


if __name__ == "__main__":
    main()


### الخطوة 24: ملف المتطلبات وملف المفاتيح النموذجي

In [ ]:
%%writefile requirements.txt
# مساعد تخطيط السفر — المكتبات المطلوبة
langchain-core>=0.3,<2       # القوالب، LCEL، المحللات، الأدوات، الذاكرة
langchain-groq>=0.2,<2       # ربط LangChain بنماذج Groq (Llama)
streamlit>=1.36              # واجهة المستخدم
pydantic>=2.7                # المخرجات المنظمة والتحقق
python-dotenv>=1.0           # قراءة المفاتيح من ملف .env
pytest>=8.0                  # الاختبارات
# sqlite3 مدمجة مع Python ولا تحتاج تثبيتاً


In [ ]:
%%writefile .env.example
# انسخ هذا الملف باسم .env وضع مفتاحك (لا ترفع ملف .env إلى أي مستودع)
GROQ_API_KEY=ضع_مفتاحك_هنا
GROQ_MODEL=openai/gpt-oss-120b
LLM_TEMPERATURE=0.3
LLM_MAX_RETRIES=2
LLM_TIMEOUT=60
# TRAVEL_DB_PATH=data/travel_planner.db


In [ ]:
%%writefile .gitignore
.env
__pycache__/
*.pyc
data/*.db
.pytest_cache/


### الخطوة 25: تشغيل الواجهة برابط عام
يُنشأ رابط `trycloudflare.com` مؤقت؛ افتحه في المتصفح. أوقف الخلية لإيقاف الواجهة.

> النسخة المنشورة الدائمة: https://salman-travel-planner.streamlit.app

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, time, re
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("🌐 رابط الواجهة:", m.group(0))
        break

## المكتبات والأدوات الخارجية المستخدمة
| الأداة | الاستخدام |
|:---:|:---:|
| LangChain Core و LangChain Groq | تنسيق النظام وربط النموذج |
| Groq API (نموذج GPT-OSS 120B) | النموذج اللغوي |
| Streamlit | واجهة المستخدم |
| Pydantic | المخرجات المنظمة والتحقق |
| SQLite (مدمجة مع Python) | قاعدة البيانات |
| python-dotenv | قراءة المفاتيح محلياً |
| pytest | الاختبارات |
| cloudflared | رابط عام مؤقت للواجهة من Colab |
| خط Tajawal من Google Fonts | خط الواجهة العربي |

> جميع التكاليف تقديرية من بيانات تجريبية للنموذج الأولي، ولا تمثل أسعاراً رسمية.